In [9]:
import sys, subprocess

PROXY = "http://httpproxy-tcop.vip.ebay.com:80"

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--user",
    "--proxy", PROXY,
    "--upgrade",
    "transformers==4.46.3",
    "huggingface_hub==0.26.2",
    "tokenizers==0.20.3",
    "accelerate==1.0.1",
    "datasets==3.1.0",
], check=True)

print("Restart kernel after install.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 146.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 151.2 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.12.0
    Uninstalling huggingface_hub-1.12.0:
      Successfully uninstalled huggingface_hub-1.12.00/5 [huggingface_hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/5 [huggingface_hub]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.20/5 [huggingface_hub]
    Uninstalling tokenizers-0.22.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [tokenizers]
      Successfully uninstalled tokenizers-0.22.2━━━━━━━━━━━━━━ 1/5 [tokenizers]
  Attempting uninstall: transformers━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [tokenizers]
    Found existing installation: transformers 5.6.2━━━━━━━━━━━ 1/5 [tokenizers]
    Uninstalling transformers-5.6.2:0m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [transformers]
      Successfully uninstalled transformers-5.6.2━━━━━━━━━━━━━━━━━ 2/5 [transformers]
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [transformers]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Attempting uninstall: accelerate━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [transformers]
    Found existing installation: accelerate 1.13.0━━━━━━━━━━━━ 2/5 [transformers]
    Uninstalling accelerate-1.13.0:m╺━━━━━━━━━━━━━━━ 3/5 [accelerate]
      Successfully uninstalled accelerate-1.13.0━━━━━━━━━━━━━━ 3/5 [accelerate]
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 3/5 [accelerate]

...

  Attempting uninstall: datasets╺━━━━━━━━━━━━━━━ 3/5 [accelerate]
    Found existing installation: datasets 4.8.4━━━━━━━━━━━━━━━ 3/5 [accelerate]
    Uninstalling datasets-4.8.4:╺━━━━━━━━━━━━━━━ 3/5 [accelerate]
      Successfully uninstalled datasets-4.8.40m━━━━━━━━━━━━━━━ 3/5 [accelerate]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [datasets]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [datasets]4/5 [datasets]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
coreai-propeller 1.0.0 requires datasets==1.9.0, but you have datasets 3.1.0 which is incompatible.
coreai-propeller 1.0.0 requires pandas<2.0.0,>=1.1.5, but you have pandas 2.3.3 which is incompatible.
coreai-propeller 1.0.0 requires pyarrow<10.0.0,>=9.0.0, but you have pyarrow 24.0.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


Restart kernel after install.


# Prompt Sensitivity Analysis Across LLM Families

v6 fixes (based on detailed feedback review):
1. Chat template: replaced hardcoded templates with tokenizer.apply_chat_template()
   - Llama had 14/50 MNLI, 10/50 BBH, 19/50 ARC disagreements with native template
   - Gemma and Qwen were fine (<5 disagreements) but now use native too for consistency
2. Removed struct_inst_last: was identical to baseline (0.0 flip_rate everywhere)
3. Replaced FI formula: old 1/(d_sem+eps) produced million-scale artifacts when d_sem=0
   - New: sensitivity_score = flip_rate × |acc_drop| (bounded, interpretable)
   - d_sem kept as separate reported metric
4. Replaced lex_synonym: nlpaug WordNet produced "Solvent" for "Answer" — broken
   - Now uses hand-audited lexical variants per task
5. Fixed para_rewrite MNLI: old version changed label names (supported/contradicted)
   - New version preserves entailment/neutral/contradiction
6. Renamed para_rewrite → instruction_rewrite for accuracy
7. Added collapse diagnostics: flags runs where one label >90% of predictions
8. Added perturbation audit CSV export (results/prompt_templates.csv)
9. Enhanced stats output: n_discordant, direction, underpowered flag
10. Base models moved to appendix framing (kept in code, flagged in output)


## Cell 1 - Install Dependencies

In [1]:
import os
import sys
import subprocess
import nltk

PROXY = "http://httpproxy-tcop.vip.ebay.com:80"

# Make proxy visible to Python libraries that use urllib/request
os.environ["http_proxy"] = PROXY
os.environ["https_proxy"] = PROXY
os.environ["HTTP_PROXY"] = PROXY
os.environ["HTTPS_PROXY"] = PROXY

# Also tell NLTK explicitly
nltk.set_proxy(PROXY)

def pip_install(*packages, index_url=None, upgrade=False, user=True):
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--proxy", PROXY,
    ]
    if user:
        cmd.append("--user")
    if upgrade:
        cmd.append("--upgrade")
    if index_url:
        cmd += ["--index-url", index_url]
    cmd += list(packages)
    subprocess.run(cmd, check=True)

# pip installs
pip_install("torch", index_url="https://download.pytorch.org/whl/cu121")

packages = [
    "transformers",
    "huggingface_hub",
    "tokenizers",
    "accelerate",
    "bitsandbytes",
    "sentence-transformers",
    "datasets",
    "scipy",
    "nlpaug",
    "pandas","numpy==1.26.4",
    "jinja2>=3.1.0",
]
pip_install(*packages, upgrade=True)

# NLTK downloads
for pkg in [
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "wordnet",
    "omw-1.4",
]:
    print(f"Downloading {pkg}...")
    nltk.download(pkg, quiet=False)

print("Done. Restart kernel before running anything else.")

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 126.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 908.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 155.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 892.9 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.26.2
    Uninstalling huggingface-hub-0.26.2:
      Successfully uninstalled huggingface-hub-0.26.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/5 [huggingface_hub]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/5 [huggingface_hub]
    Found existing installation: tokenizers 0.20.30/5 [huggingface_hub]
    Uninstalling tokenizers-0.20.3:━━━━━━━━━ 0/5 [huggingface_hub]
      Successfully uninstalled tokenizers-0.20.3 0/5 [huggingface_hub]
  Attempting uninstall: datasets90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [tokenizers]
    Found existing installation: datasets 3.1.0━━━━━━━━━━━━━━━ 1/5 [tokenizers]
    Uninstalling datasets-3.1.0:m╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [datasets]
      Successfully uninstalled datasets-3.1.0━━━━━━━━━━━━━━━━━ 2/5 [datasets]
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [datasets]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Attempting uninstall: accelerate━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [datasets]
    Found existing installation: accelerate 1.0.1━━━━━━━━━━━━━ 2/5 [datasets]
    Uninstalling accelerate-1.0.1:━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [datasets]
      Successfully uninstalled accelerate-1.0.1━━━━━━━━━━━━━━━ 3/5 [accelerate]
   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 3/5 [accelerate]

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformers] [transformers]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
coreai-propeller 1.0.0 requires datasets==1.9.0, but you have datasets 4.8.4 which is incompatible.
coreai-propeller 1.0.0 requires pandas<2.0.0,>=1.1.5, but you have pandas 2.3.3 which is incompatible.
coreai-propeller 1.0.0 requires pyarrow<10.0.0,>=9.0.0, but you have pyarrow 24.0.0 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


[nltk_data] Error loading averaged_perceptron_tagger: <urlopen error
[nltk_data]     Tunnel connection failed: 407 Proxy Authentication
[nltk_data]     Required>
[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error Tunnel connection failed: 407 Proxy
[nltk_data]     Authentication Required>
[nltk_data] Error loading wordnet: <urlopen error Tunnel connection
[nltk_data]     failed: 407 Proxy Authentication Required>


Done. Restart kernel before running anything else.


[nltk_data] Error loading omw-1.4: <urlopen error Tunnel connection
[nltk_data]     failed: 407 Proxy Authentication Required>


## Cell 2 - Version Check

In [1]:
import transformers, huggingface_hub, tokenizers, accelerate, torch, bitsandbytes

print(f"torch:           {torch.__version__}")
print(f"transformers:    {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"bitsandbytes:    {bitsandbytes.__version__}")
print(f"tokenizers:      {tokenizers.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch:           2.6.0+cu126
transformers:    5.6.2
huggingface_hub: 1.12.0
bitsandbytes:    0.49.2
tokenizers:      0.22.2
accelerate:      1.13.0
CUDA available:  True
GPU:             NVIDIA H100 80GB HBM3
VRAM:            85.2 GB


## Cell 3 - HuggingFace Authentication

In [ ]:
import os

PROXY = "http://httpproxy-tcop.vip.ebay.com:80"
os.environ["http_proxy"] = PROXY
os.environ["https_proxy"] = PROXY
os.environ["HTTP_PROXY"] = PROXY
os.environ["HTTPS_PROXY"] = PROXY

os.environ["HF_TOKEN"] = ""
print("Proxy and HF token configured.")


Proxy and HF token configured.


## Cell 4 - Imports and Configuration

[MUDIT] Seeds fixed at module level. They matter for the one-time cache build and
dataset sampling. After the cache is on disk they have no further effect on results.

[REVIEW Fix 8] Dataset sampling uses an isolated numpy RNG (default_rng) so it cannot be
clobbered by library calls that touch the global numpy random state.

[NEHA] Gemma-2-9B base added to MODEL_REGISTRY for instruct vs base comparison.

[ALL] N_MNLI and N_ARC raised to 300. BBH capped at 250 (full dataset).

[v6] chat_template field in MODEL_REGISTRY now stores "auto" for instruct models
instead of hardcoded template names. format_prompt uses tokenizer.apply_chat_template.


In [13]:
import torch
import gc
import json
import csv
import os
import re
import time
import random
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from collections import defaultdict

# [MUDIT] Fix all seeds before anything runs.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# [REVIEW Fix 8] Isolated RNG for dataset sampling.
_sample_rng = np.random.default_rng(SEED)

print(f"Seeds fixed: {SEED}")

N_MNLI = 300
N_BBH  = 250  # full dataset
N_ARC  = 300
print(f"Example counts: MNLI={N_MNLI}, BBH={N_BBH}, ARC={N_ARC}")

QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# [v6] chat_template="auto" means use tokenizer.apply_chat_template
# chat_template=None means base model, use raw prompt
MODEL_REGISTRY = {
    "llama3.1-8b-instruct": {
        "hf_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "family": "llama", "type": "instruct", "params": "8B", "chat_template": "auto",
    },
    "gemma2-9b-it": {
        "hf_id": "google/gemma-2-9b-it",
        "family": "gemma", "type": "instruct", "params": "9B", "chat_template": "auto",
    },
    "qwen2.5-7b-instruct": {
        "hf_id": "Qwen/Qwen2.5-7B-Instruct",
        "family": "qwen", "type": "instruct", "params": "7B", "chat_template": "auto",
    },
    "llama3.1-8b-base": {
        "hf_id": "meta-llama/Meta-Llama-3.1-8B",
        "family": "llama", "type": "base", "params": "8B", "chat_template": None,
    },
    "gemma2-9b-base": {
        "hf_id": "google/gemma-2-9b",
        "family": "gemma", "type": "base", "params": "9B", "chat_template": None,
    },
    "qwen2.5-7b-base": {
        "hf_id": "Qwen/Qwen2.5-7B",
        "family": "qwen", "type": "base", "params": "7B", "chat_template": None,
    },
}

os.makedirs("results/paired", exist_ok=True)

print("Configuration loaded.")
for name, info in MODEL_REGISTRY.items():
    tpl = info["chat_template"] or "raw"
    print(f"  {name:<28} [{info['type']}] template={tpl}")


# [v6.1] Perturbation names defined here (before cache cell needs them)
ALL_PERT_NAMES = [
    "baseline",
    "lex_rewrite", "lex_typo", "lex_lowercase",
    "struct_inst_first",
    "ctx_role", "ctx_irrelevant",
    "fmt_caps", "fmt_separators",
    "instruction_rewrite",
]

PERT_CATEGORIES = {
    "baseline":            "baseline",
    "lex_rewrite":         "lexical",
    "lex_typo":            "lexical",
    "lex_lowercase":       "lexical",
    "struct_inst_first":   "structural",
    "ctx_role":            "contextual",
    "ctx_irrelevant":      "contextual",
    "fmt_caps":            "formatting",
    "fmt_separators":      "formatting",
    "instruction_rewrite": "rewrite",
}

print(f"Perturbation types: {len(ALL_PERT_NAMES)} ({len(ALL_PERT_NAMES)-1} + baseline)")


Seeds fixed: 42
Example counts: MNLI=300, BBH=250, ARC=300
Configuration loaded.
  llama3.1-8b-instruct         [instruct] template=auto
  gemma2-9b-it                 [instruct] template=auto
  qwen2.5-7b-instruct          [instruct] template=auto
  llama3.1-8b-base             [base] template=raw
  gemma2-9b-base               [base] template=raw
  qwen2.5-7b-base              [base] template=raw
Perturbation types: 10 (9 + baseline)


## Cell 5 - Model Loading

In [14]:
_current_model = None
_current_tokenizer = None
_current_model_name = None


def clear_model():
    global _current_model, _current_tokenizer, _current_model_name
    if _current_model is not None:
        print(f"Clearing {_current_model_name} from memory.")
        del _current_model
        del _current_tokenizer
        _current_model = None
        _current_tokenizer = None
        _current_model_name = None
        gc.collect()
        torch.cuda.empty_cache()
        print(f"GPU freed. Available: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
    else:
        print("No model loaded.")


def load_model(model_key: str):
    global _current_model, _current_tokenizer, _current_model_name
    if model_key == _current_model_name:
        print(f"{model_key} already loaded.")
        return _current_model, _current_tokenizer
    if model_key not in MODEL_REGISTRY:
        raise ValueError(f"Unknown key: {model_key}. Options: {list(MODEL_REGISTRY.keys())}")
    clear_model()
    info = MODEL_REGISTRY[model_key]
    hf_id = info["hf_id"]
    print(f"Loading {model_key} from {hf_id}")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        quantization_config=QUANT_CONFIG,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model.eval()
    elapsed = time.time() - start
    vram = (torch.cuda.mem_get_info()[1] - torch.cuda.mem_get_info()[0]) / 1e9
    print(f"Loaded in {elapsed:.0f}s, VRAM used: {vram:.1f} GB")
    _current_model = model
    _current_tokenizer = tokenizer
    _current_model_name = model_key
    return model, tokenizer


print("Model management functions ready.")

Model management functions ready.


## Cell 6 - Inference Pipeline

[v6.1] format_and_tokenize uses apply_chat_template(tokenize=True) for instruct models.
This avoids the double-BOS problem: apply_chat_template(tokenize=False) returns a string
with special tokens already embedded, but tokenizer() adds them again by default.
Using tokenize=True directly is the HuggingFace-recommended approach.

Base models still go through normal tokenizer() since they have no chat template.

format_prompt() is kept as a convenience for logging/display only — never for tokenization.


[v7] Fix C1+C2: Added continuation_token_ids() helper for correct label scoring.
The old approach used tokenizer.encode(" " + label)[0] which produces the space-prefixed
token, but after apply_chat_template the model generates bare-letter continuations.
Both get_label_logprobs() and verify_single_token_labels() now use the same helper.


In [15]:
@dataclass
class InferenceResult:
    predicted_label: str
    label_logprobs: Dict[str, float]
    correct: bool
    gold_label: str
    prompt: str
    margin: float  # top logprob minus second logprob (confidence)


def _extract_input_ids_tensor(maybe_ids) -> torch.Tensor:
    """
    Normalize tokenizer / apply_chat_template outputs to a rank-2 input_ids tensor.

    HF versions differ here:
    - some return a plain torch.Tensor
    - some return a BatchEncoding / dict with "input_ids"

    This prevents model(..., input_ids=<BatchEncoding>) crashes.
    """
    if torch.is_tensor(maybe_ids):
        input_ids = maybe_ids
    elif hasattr(maybe_ids, "input_ids"):
        input_ids = maybe_ids.input_ids
    elif isinstance(maybe_ids, dict) and "input_ids" in maybe_ids:
        input_ids = maybe_ids["input_ids"]
    else:
        raise TypeError(
            f"Expected tensor or BatchEncoding-like object with input_ids, got {type(maybe_ids)}"
        )

    if not torch.is_tensor(input_ids):
        raise TypeError(f"input_ids must be a torch.Tensor, got {type(input_ids)}")
    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)
    return input_ids


def _common_prefix_len(a: List[int], b: List[int]) -> int:
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def label_variant_token_ids(tokenizer, label: str) -> List[int]:
    """
    For base-model scoring, accept a small set of immediate 1-token variants
    that correspond to the same answer label.
    """
    variants = []
    for text in (label, " " + label, "\n" + label):
        ids = tokenizer.encode(text, add_special_tokens=False)
        if len(ids) == 1:
            variants.append(ids[0])
    return sorted(set(variants))


def continuation_token_ids(
    tokenizer,
    label: str,
    prompt_ids: List[int] = None,
    prompt_str: str = None,
) -> List[int]:
    """
    Return token ID(s) for `label` as a continuation of the prompt.

    Modes
    -----
    1) Strict scoring mode (prompt_ids + prompt_str provided):
       - used for instruct models
       - encodes prompt_str + label with add_special_tokens=False
       - asserts the encoded sequence begins with prompt_ids exactly
       - returns the suffix
       - raises on prefix mismatch or multi-token continuations

    2) Probe mode (prompt_ids=None):
       - used only for lightweight preflight checks
       - computes a longest-common-prefix diff
       - does NOT silently fall back to guessed token IDs
    """
    if prompt_ids is not None and prompt_str is not None:
        prompt_ids = list(prompt_ids)
        full_ids = tokenizer.encode(prompt_str + label, add_special_tokens=False)
        prompt_len = len(prompt_ids)

        if len(full_ids) <= prompt_len:
            raise RuntimeError(
                f"continuation_token_ids: encoding prompt+{label!r} produced "
                f"{len(full_ids)} tokens, not more than prompt's {prompt_len}. "
                f"Label was absorbed into the prompt boundary."
            )

        if full_ids[:prompt_len] != prompt_ids:
            raise RuntimeError(
                f"continuation_token_ids: prefix mismatch for label {label!r}. "
                f"prompt_ids[-6:]={prompt_ids[-6:]} vs "
                f"full_ids[:prompt_len][-6:]={full_ids[max(0, prompt_len-6):prompt_len]}. "
                f"The tokenizer re-segmented the boundary."
            )

        continuation = full_ids[prompt_len:]
        if len(continuation) != 1:
            raise RuntimeError(
                f"continuation_token_ids: label {label!r} is {len(continuation)} continuation "
                f"tokens in this prompt context: {continuation}. "
                f"Single-token label scoring is invalid for this example."
            )
        return continuation

    ctx = prompt_str if prompt_str is not None else "Answer:\n"
    ids_without = tokenizer.encode(ctx, add_special_tokens=False)
    ids_with = tokenizer.encode(ctx + label, add_special_tokens=False)
    common = _common_prefix_len(ids_without, ids_with)
    return ids_with[common:]


def format_and_tokenize(model_key: str, user_message: str, tokenizer=None):
    """
    Returns tokenized input_ids as a tensor, ready for model forward pass.

    For instruct models: uses apply_chat_template(tokenize=True) directly.
    For base models: tokenizes the raw prompt normally.

    Also returns the prompt string (for logging/storage) via a separate
    tokenize=False call for instruct models.
    """
    info = MODEL_REGISTRY[model_key]
    template = info["chat_template"]

    if template is None:
        encoded = tokenizer(user_message, return_tensors="pt")
        input_ids = _extract_input_ids_tensor(encoded)
        return input_ids, user_message

    if tokenizer is None:
        raise ValueError(f"tokenizer required for instruct model {model_key}")

    messages = [{"role": "user", "content": user_message}]

    templated = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    input_ids = _extract_input_ids_tensor(templated)

    prompt_str = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return input_ids, prompt_str


# Backward-compatible wrapper for code that still calls format_prompt
def format_prompt(model_key: str, user_message: str, tokenizer=None) -> str:
    """Returns the prompt string only (for display/logging). Not for tokenization."""
    info = MODEL_REGISTRY[model_key]
    template = info["chat_template"]
    if template is None:
        return user_message
    if tokenizer is None:
        raise ValueError(f"tokenizer required for instruct model {model_key}")
    messages = [{"role": "user", "content": user_message}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def get_label_logprobs(
    model,
    tokenizer,
    model_key: str,
    user_message: str,
    candidate_labels: List[str],
    gold_label: str,
) -> InferenceResult:
    """
    Label scoring:

    - Instruct models:
        strict prompt-aware continuation extraction using the exact prompt IDs
        the model saw.

    - Base models:
        raw prompts do not provide a stable prefix assertion boundary, so score
        each label by log-summing a small set of immediate 1-token variants
        (e.g. "A", " A", "\\nA") instead of crashing or guessing only one token.
    """
    input_ids, prompt_str = format_and_tokenize(model_key, user_message, tokenizer=tokenizer)
    input_ids = _extract_input_ids_tensor(input_ids)
    model_input_ids = input_ids.to(model.device)

    with torch.no_grad():
        outputs = model(input_ids=model_input_ids, use_cache=False)
        next_token_logits = outputs.logits[0, -1, :]
        log_probs = torch.log_softmax(next_token_logits, dim=-1)

    info = MODEL_REGISTRY[model_key]
    label_logprobs = {}

    if info["chat_template"] is not None:
        # Instruct: strict prompt-aware scoring
        prompt_ids_for_assert = input_ids[0].tolist()

        for label in candidate_labels:
            cont_ids = continuation_token_ids(
                tokenizer,
                label,
                prompt_ids=prompt_ids_for_assert,
                prompt_str=prompt_str,
            )
            label_logprobs[label] = log_probs[cont_ids[0]].item()
    else:
        # Base: aggregate over plausible immediate 1-token label variants
        for label in candidate_labels:
            variant_ids = label_variant_token_ids(tokenizer, label)
            if not variant_ids:
                tried_variants = [label, " " + label, "\n" + label]
                tried_str = ", ".join(repr(v) for v in tried_variants)
                raise RuntimeError(
                    f"{model_key}: no 1-token immediate variants found for label {label!r}. "
                    f"Tried {tried_str}."
                )
            idx = torch.tensor(variant_ids, device=log_probs.device, dtype=torch.long)
            label_logprobs[label] = torch.logsumexp(log_probs.index_select(0, idx), dim=0).item()

    predicted_label = max(label_logprobs, key=label_logprobs.get)
    sorted_lp = sorted(label_logprobs.values(), reverse=True)
    margin = sorted_lp[0] - sorted_lp[1] if len(sorted_lp) >= 2 else 0.0

    return InferenceResult(
        predicted_label=predicted_label,
        label_logprobs=label_logprobs,
        correct=(predicted_label == gold_label),
        gold_label=gold_label,
        prompt=prompt_str,
        margin=margin,
    )


print("Inference pipeline ready (instruct: strict prompt-aware scoring; base: 1-token label-variant aggregation).")

Inference pipeline ready (instruct: strict prompt-aware scoring; base: 1-token label-variant aggregation).


## Cell 7 - Load Datasets

[VISHNU] MNLI gold labels stored as A/B/C. BBH candidate labels dynamic per example.
ARC candidate labels from each example's choices field.

[REVIEW Fix 5] ARC numeric label detection and remapping. Some ARC items use
"1","2","3","4" as labels. These are not reliably single-token across all tokenizers.
When detected, they are remapped to A,B,C,D in choices_text, candidate_labels, and gold_label.

[REVIEW Fix 8] Sampling uses _sample_rng (isolated from global numpy state).

In [16]:
from datasets import load_dataset

MNLI_LABEL_MAP = {"entailment": "A", "neutral": "B", "contradiction": "C"}

print("Loading MNLI...")
mnli_raw = load_dataset("nyu-mll/multi_nli", split="validation_matched")
int_label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
mnli_by_label = defaultdict(list)
for ex in mnli_raw:
    label_str = int_label_map[ex["label"]]
    mnli_by_label[label_str].append({
        "premise":    ex["premise"],
        "hypothesis": ex["hypothesis"],
        "gold_label": MNLI_LABEL_MAP[label_str],
        "task":       "mnli",
    })

# [REVIEW Fix 8] Shuffle each bucket with isolated RNG before drawing N//3
mnli_data = []
for label_str, bucket in mnli_by_label.items():
    idx = _sample_rng.permutation(len(bucket))[:N_MNLI // 3]
    mnli_data.extend([bucket[i] for i in idx])
# [v7.1 Fix D1] Shuffle so mnli_data isn't grouped A,A,...B,B,...C,C.
# Without this, sanity checks on data[:50] only test label A.
_sample_rng.shuffle(mnli_data)
print(f"  MNLI: {len(mnli_data)} examples (shuffled), labels: {set(ex['gold_label'] for ex in mnli_data)}")
print(f"    First 6 labels: {[ex['gold_label'] for ex in mnli_data[:6]]}")


def get_bbh_labels(input_text: str) -> list:
    matches = re.findall(r'\(([A-F])\)', input_text)
    seen, result = set(), []
    for m in matches:
        if m not in seen:
            seen.add(m)
            result.append(m)
    return result if result else ["A", "B", "C", "D", "E"]

print("Loading BBH date_understanding...")
bbh_raw = load_dataset("lukaemon/bbh", "date_understanding", split="test")
bbh_all = []
for ex in bbh_raw:
    answer_raw = ex["target"].strip()
    answer_letter = answer_raw.strip("()") if answer_raw.startswith("(") else answer_raw[0]
    bbh_all.append({
        "input":            ex["input"],
        "gold_label":       answer_letter,
        "candidate_labels": get_bbh_labels(ex["input"]),
        "task":             "bbh_date",
    })
# [REVIEW Fix 8] Seeded sample from full BBH set
bbh_idx = _sample_rng.choice(len(bbh_all), size=min(N_BBH, len(bbh_all)), replace=False)
bbh_data = [bbh_all[i] for i in sorted(bbh_idx)]
print(f"  BBH: {len(bbh_data)} examples")


def remap_arc_numeric_labels(choices_labels, gold_label):
    """
    [v7.1 Fix D2] Some ARC items use 1/2/3/4 as option labels instead of A/B/C/D.
    Remaps labels and gold BEFORE choices_text is constructed, so .replace()
    never runs on answer text (which could contain "(1)" in explanations).
    Returns (new_labels, new_gold).
    """
    if not choices_labels or not choices_labels[0].isdigit():
        return choices_labels, gold_label
    num_to_letter = {"1": "A", "2": "B", "3": "C", "4": "D", "5": "E"}
    new_labels = [num_to_letter.get(l, l) for l in choices_labels]
    new_gold = num_to_letter.get(gold_label, gold_label)
    return new_labels, new_gold

print("Loading ARC-Challenge...")
arc_raw = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
arc_all = []
for ex in arc_raw:
    # [v7.1 Fix D2] Remap numeric labels BEFORE constructing choices_text
    # so .replace() never touches answer text that might contain "(1)" etc.
    raw_labels = list(ex["choices"]["label"])
    raw_gold = ex["answerKey"]
    c_labels, gold = remap_arc_numeric_labels(raw_labels, raw_gold)

    # Build choices_text with already-remapped labels
    choices_text = "\n".join(
        f"({label}) {text}"
        for label, text in zip(c_labels, ex["choices"]["text"])
    )
    arc_all.append({
        "question":         ex["question"],
        "choices_text":     choices_text,
        "candidate_labels": c_labels,
        "gold_label":       gold,
        "task":             "arc",
    })
# [REVIEW Fix 8] Seeded sample
arc_idx = _sample_rng.choice(len(arc_all), size=min(N_ARC, len(arc_all)), replace=False)
arc_data = [arc_all[i] for i in sorted(arc_idx)]
print(f"  ARC: {len(arc_data)} examples")

# Ensure all ARC labels are uppercase letters after remapping.
# Force-uppercase any lowercase labels rather than crashing.
fixed_count = 0
for ex in arc_data:
    ex["candidate_labels"] = [l.upper() if l.isalpha() else l for l in ex["candidate_labels"]]
    if ex["gold_label"].isalpha() and not ex["gold_label"].isupper():
        ex["gold_label"] = ex["gold_label"].upper()
        fixed_count += 1

bad_arc = [ex for ex in arc_data if any(not l.isupper() or not l.isalpha() for l in ex["candidate_labels"])]
if bad_arc:
    print(f"  WARNING: {len(bad_arc)} ARC examples have non-letter labels after remapping: {bad_arc[:2]}")
else:
    print(f"  ARC label check passed: all labels are uppercase letters (fixed {fixed_count}).")

print(f"\nTotal: {len(mnli_data)+len(bbh_data)+len(arc_data)} examples")

Loading MNLI...


  MNLI: 300 examples (shuffled), labels: {'A', 'B', 'C'}
    First 6 labels: ['A', 'A', 'B', 'C', 'C', 'C']
Loading BBH date_understanding...
  BBH: 250 examples
Loading ARC-Challenge...
  ARC: 300 examples
  ARC label check passed: all labels are uppercase letters (fixed 0).

Total: 850 examples


## Cell 8 - Prompt Templates (Body / Instruction Split)

[REVIEW Fix 1] Prompts are now split into two parts:
- body: the task content (premise/hypothesis, question, choices) — never perturbed
- instruction: the directive text at the end — the only part that perturbations touch

This ensures lexical/formatting perturbations affect the prompt wrapper only, not the
task content. Previously perturb_synonym() was run on the full prompt including premises
and hypotheses, which made it input corruption rather than prompt fragility.

make_*_prompt() wrappers remain for backwards compatibility and for the cache builder.

In [17]:
def mnli_parts(ex: dict) -> Tuple[str, str]:
    """Returns (body, instruction) for MNLI example."""
    body = f"Premise: {ex['premise']}\nHypothesis: {ex['hypothesis']}"
    instruction = (
        "Does the premise entail, contradict, or is it neutral to the hypothesis?\n"
        "Answer with one letter only: A for entailment, B for neutral, C for contradiction."
    )
    return body, instruction


def bbh_parts(ex: dict) -> Tuple[str, str]:
    body = ex["input"]
    instruction = "Choose the best answer. Respond with only the letter."
    return body, instruction


def arc_parts(ex: dict) -> Tuple[str, str]:
    # ARC numeric labels (1/2/3/4) are remapped to A/B/C/D in Cell 7.
    # By the time arc_parts() is called, all candidate_labels are uppercase letters.
    # The instruction saying 'letter' is therefore always correct.
    # This is verified by the assertion in the dataset loading cell.
    body = f"Question: {ex['question']}\n{ex['choices_text']}"
    instruction = "Answer with only the letter of the correct choice."
    return body, instruction


def join_prompt(body: str, instruction: str) -> str:
    return f"{body}\n{instruction}"


# Dispatch dict used by perturbation engine
PARTS_FN = {
    "mnli":     mnli_parts,
    "bbh_date": bbh_parts,
    "arc":      arc_parts,
}

# Convenience wrappers — used in cache build and sanity checks
def make_mnli_prompt(ex): body, instr = mnli_parts(ex); return join_prompt(body, instr)
def make_bbh_prompt(ex):  body, instr = bbh_parts(ex);  return join_prompt(body, instr)
def make_arc_prompt(ex):  body, instr = arc_parts(ex);  return join_prompt(body, instr)

print("Sample MNLI prompt:")
print(make_mnli_prompt(mnli_data[0]))
print("\nSample BBH prompt (first 400 chars):")
print(make_bbh_prompt(bbh_data[0])[:400])
print("\nSample ARC prompt:")
print(make_arc_prompt(arc_data[0]))

Sample MNLI prompt:
Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated in as least one way. 
Does the premise entail, contradict, or is it neutral to the hypothesis?
Answer with one letter only: A for entailment, B for neutral, C for contradiction.

Sample BBH prompt (first 400 chars):
Today is Christmas Eve of 1937. What is the date tomorrow in MM/DD/YYYY?
Options:
(A) 12/11/1937
(B) 12/25/1937
(C) 01/04/1938
(D) 12/04/1937
(E) 12/25/2006
(F) 07/25/1937
Choose the best answer. Respond with only the letter.

Sample ARC prompt:
Question: A group of engineers wanted to know how different building designs would respond during an earthquake. They made several models of buildings and tested each for its ability to withstand earthquake conditions. Which will most likely result from testing different building designs?
(A) buildings will be built faster
(B) buildings will be made safer
(C) buildin

## Cell 9 - Perturbation Engine

[v6] Major changes:
1. Removed struct_inst_last (identical to baseline — 0.0 flip rate everywhere)
2. Replaced lex_synonym: nlpaug WordNet produced "Solvent" for "Answer" etc.
   Now uses hand-audited lexical variants per task that preserve label mappings.
3. Renamed para_rewrite → instruction_rewrite for accuracy.
4. Fixed MNLI instruction_rewrite: preserves entailment/neutral/contradiction
   (old version changed to supported/neither/contradicted which alters semantics).
5. All lexical perturbations still operate on instruction only, not body.


In [18]:
import nlpaug.augmenter.char as nac
import re
import random

_typo_aug = nac.RandomCharAug(action="substitute", aug_char_p=0.05)


# [v6] Hand-audited lexical variants per task.
# Rules: (1) change wording, (2) keep label mappings identical, (3) keep output format constraint.
LEXICAL_VARIANTS = {
    "mnli": (
        "What is the relationship between the premise and the hypothesis?\n"
        "Answer with one letter only: A for entailment, B for neutral, C for contradiction."
    ),
    "bbh_date": "Select the best answer. Respond with only the letter.",
    "arc": "Select the correct choice and answer with only the letter.",
}


def perturb_lexical(instruction: str, task: str) -> str:
    """[v6] Returns a hand-audited lexical variant that preserves label mappings."""
    return LEXICAL_VARIANTS.get(task, instruction)


def perturb_typo(instruction: str) -> str:
    # No per-call seed reset — seed is set once in cache builder.
    # This allows different instructions to get different typo patterns.
    try:
        return _typo_aug.augment(instruction)[0]
    except Exception:
        return instruction


def perturb_formality_lower(instruction: str) -> str:
    # Lowercase instruction but restore uppercase answer letters (A-F).
    # Uses context-aware regex: only capitalize letters that appear as answer
    # labels (after "for ", in parentheses, after colon, etc.), not the
    # English article "a" in phrases like "choose a letter".
    lowered = instruction.lower()
    # Match label patterns: "(a)", "a for", "a:", standalone at line boundary
    lowered = re.sub(
        r'(?:(?<=\()|(?<=\bfor\s)|(?<=:\s)|(?<=,\s))([a-f])(?=\)|\s+for\b|[,.\s]|$)',
        lambda m: m.group(1).upper(),
        lowered
    )
    # Fallback: also restore any "a for entailment" / "b for neutral" patterns
    lowered = re.sub(r'\b([a-f])\s+for\s', lambda m: m.group().replace(m.group(1), m.group(1).upper()), lowered)
    return lowered


def perturb_role_preamble(instruction: str) -> str:
    return "You are an expert linguist and logician. " + instruction


def perturb_irrelevant_context(instruction: str) -> str:
    return "Note: This is part of a larger evaluation suite. " + instruction


def perturb_formatting_caps(instruction: str) -> str:
    return instruction.upper()


def perturb_formatting_separators(body: str, instruction: str) -> str:
    return join_prompt(body, instruction).replace("\n", " | ")


def get_all_perturbations(ex: dict, task: str) -> list:
    """
    Returns list of 4-tuples: (name, category, full_prompt, instruction_used)

    [v6] Changes from v5:
    - lex_synonym replaced with lex_rewrite (hand-audited per task)
    - struct_inst_last removed (was identical to baseline)
    - para_rewrite renamed to instruction_rewrite with fixed MNLI labels
    """
    body, base_instr = PARTS_FN[task](ex)
    base_prompt = join_prompt(body, base_instr)

    perts = [("baseline", "baseline", base_prompt, base_instr)]

    # Lexical — perturb instruction only, rejoin with unchanged body
    lex_instr  = perturb_lexical(base_instr, task)
    typo_instr = perturb_typo(base_instr)
    low_instr  = perturb_formality_lower(base_instr)
    perts.append(("lex_rewrite",   "lexical", join_prompt(body, lex_instr),  lex_instr))
    perts.append(("lex_typo",      "lexical", join_prompt(body, typo_instr), typo_instr))
    perts.append(("lex_lowercase", "lexical", join_prompt(body, low_instr),  low_instr))

    # Structural — instruction BEFORE body (non-standard order)
    # [v6] struct_inst_last removed: it was body\ninstruction = same as baseline
    first_prompt = f"{base_instr}\n{body}"
    perts.append(("struct_inst_first", "structural", first_prompt, base_instr))

    # Contextual — perturb instruction only
    role_instr = perturb_role_preamble(base_instr)
    irr_instr  = perturb_irrelevant_context(base_instr)
    perts.append(("ctx_role",       "contextual", join_prompt(body, role_instr), role_instr))
    perts.append(("ctx_irrelevant", "contextual", join_prompt(body, irr_instr),  irr_instr))

    # Formatting
    caps_instr = perturb_formatting_caps(base_instr)
    sep_prompt = perturb_formatting_separators(body, base_instr)
    perts.append(("fmt_caps",       "formatting", join_prompt(body, caps_instr), caps_instr))
    perts.append(("fmt_separators", "formatting", sep_prompt,                    base_instr))

    # Instruction rewrite (manual semantic-preserving rewrite)
    # [v6] MNLI: preserves entailment/neutral/contradiction (old version changed label names)
    if task == "mnli":
        rewrite_instr = (
            "Determine the relation between the premise and the hypothesis.\n"
            "Answer with one letter only: A for entailment, B for neutral, C for contradiction."
        )
    elif task == "bbh_date":
        rewrite_instr = "Select the correct option. Write only the corresponding letter."
    else:  # arc
        rewrite_instr = "Which option is correct? Write just the letter."
    perts.append(("instruction_rewrite", "rewrite", join_prompt(body, rewrite_instr), rewrite_instr))

    return perts


print("Perturbation engine ready.")
print(f"Perturbations per example: {len(get_all_perturbations(mnli_data[0], 'mnli'))}")

# Quick check: lexical rewrite preserves label mappings
body0, instr0 = mnli_parts(mnli_data[0])
lex_instr0 = perturb_lexical(instr0, "mnli")
print(f"\nOriginal instruction:\n  {instr0}")
print(f"Lexical rewrite:\n  {lex_instr0}")
print(f"Labels preserved: {'A for entailment' in lex_instr0 and 'B for neutral' in lex_instr0}")
print(f"Body unchanged: {body0 in join_prompt(body0, lex_instr0)}")


Perturbation engine ready.
Perturbations per example: 10

Original instruction:
  Does the premise entail, contradict, or is it neutral to the hypothesis?
Answer with one letter only: A for entailment, B for neutral, C for contradiction.
Lexical rewrite:
  What is the relationship between the premise and the hypothesis?
Answer with one letter only: A for entailment, B for neutral, C for contradiction.
Labels preserved: True
Body unchanged: True


## Cell 10 - Perturbation Cache

[MUDIT] Precompute all perturbations once, save to disk. Subsequent runs load from file.

[v6] Cache auto-detects and rebuilds if:
1. Old format (prompt-only, no instruction field)
2. Old perturbation names (lex_synonym, struct_inst_last, para_rewrite)
New cache uses: lex_rewrite, struct_inst_first only, instruction_rewrite.


In [19]:
import hashlib

CACHE_DIR = "perturbation_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CACHE_FORMAT_VERSION = 2


def _seed_cache_rng():
    random.seed(SEED)
    np.random.seed(SEED)


def _stable_json_dumps(obj) -> str:
    def _default(x):
        if isinstance(x, (set, tuple)):
            return list(x)
        if isinstance(x, np.integer):
            return int(x)
        if isinstance(x, np.floating):
            return float(x)
        return str(x)

    return json.dumps(
        obj,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
        default=_default,
    )


def _materialize_cache_entries(data: list, task: str) -> dict:
    """
    Build the current cache entries from the current dataset + current perturbation code.

    This is intentionally done every session because it is cheap relative to model
    inference and guarantees the in-memory cache always matches the current notebook.
    """
    _seed_cache_rng()
    entries = {}

    for i, ex in enumerate(data):
        perts = get_all_perturbations(ex, task)
        entries[i] = {
            name: {"prompt": full_prompt, "instruction": instruction}
            for name, _cat, full_prompt, instruction in perts
        }
        if (i + 1) % 50 == 0:
            print(f"    {i+1}/{len(data)} done")

    return entries


def _cache_fingerprint(entries: dict, task: str) -> str:
    """
    Fingerprint the actual prompt/instruction cache content for this task.

    If dataset order, label mapping, prompt templates, or perturbation outputs change,
    this hash changes too.
    """
    h = hashlib.sha256()

    header = {
        "format_version": CACHE_FORMAT_VERSION,
        "task": task,
        "n_examples": len(entries),
        "seed": SEED,
        "pert_names": list(ALL_PERT_NAMES),
    }
    h.update(_stable_json_dumps(header).encode("utf-8"))

    for i in range(len(entries)):
        h.update(str(i).encode("utf-8"))
        h.update(_stable_json_dumps(entries[i]).encode("utf-8"))

    return h.hexdigest()


def _load_cache_payload(cache_path: str):
    if not os.path.exists(cache_path):
        return None

    try:
        with open(cache_path, encoding="utf-8") as f:
            raw = json.load(f)
    except Exception as e:
        print(f"  WARNING: could not read {cache_path}: {e}")
        return {"meta": {"format_version": 0, "read_error": str(e)}, "entries": None}

    # New wrapped format
    if isinstance(raw, dict) and "__meta__" in raw and "entries" in raw:
        try:
            entries = {int(k): v for k, v in raw["entries"].items()}
        except Exception:
            entries = None
        return {"meta": raw.get("__meta__", {}), "entries": entries}

    # Legacy direct format: {"0": {...}, "1": {...}, ...}
    if isinstance(raw, dict):
        try:
            entries = {int(k): v for k, v in raw.items()}
            return {"meta": {"format_version": 1, "legacy": True}, "entries": entries}
        except Exception:
            pass

    return {"meta": {"format_version": 0, "legacy": True}, "entries": None}


def build_or_load_cache(data: list, task: str, cache_path: str) -> dict:
    """
    Materialize the current cache in memory, compare it to the on-disk snapshot,
    and refresh the file if the snapshot is stale.

    Returned value is always the current in-memory cache, never blindly trusted
    from disk. This prevents stale-cache runs after notebook changes.
    """
    print(f"  Materializing current cache for {task} ({len(data)} examples)...")
    current_entries = _materialize_cache_entries(data, task)
    current_fp = _cache_fingerprint(current_entries, task)

    stored = _load_cache_payload(cache_path)
    should_write = True

    if stored is None:
        print(f"  No existing cache file: {cache_path}")
    else:
        stored_meta = stored.get("meta", {})
        stored_fp = stored_meta.get("fingerprint")
        stored_fmt = stored_meta.get("format_version", 0)

        if stored_fmt == CACHE_FORMAT_VERSION and stored_fp == current_fp:
            print(f"  Cache matches current dataset/prompts: {cache_path}")
            should_write = False
        else:
            print(f"  Cache stale or legacy; refreshing: {cache_path}")
            if stored_fp:
                print(f"    stored fingerprint:  {stored_fp[:12]}...")
            print(f"    current fingerprint: {current_fp[:12]}...")

    if should_write:
        payload = {
            "__meta__": {
                "format_version": CACHE_FORMAT_VERSION,
                "task": task,
                "n_examples": len(current_entries),
                "seed": SEED,
                "pert_names": list(ALL_PERT_NAMES),
                "fingerprint": current_fp,
            },
            "entries": {str(k): v for k, v in current_entries.items()},
        }
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False)
        print(f"  Cache saved: {cache_path}")

    return current_entries


def get_cached_entry(cache: dict, idx: int, pert_name: str) -> dict:
    """Returns {"prompt": str, "instruction": str} for one example/perturbation."""
    return cache[idx][pert_name]


print("Building/loading caches...")
print("NOTE: cache files are now self-invalidating via fingerprint checks.\n")

mnli_cache = build_or_load_cache(mnli_data[:N_MNLI], "mnli",     f"{CACHE_DIR}/mnli_cache.json")
bbh_cache  = build_or_load_cache(bbh_data[:N_BBH],  "bbh_date", f"{CACHE_DIR}/bbh_cache.json")
arc_cache  = build_or_load_cache(arc_data[:N_ARC],  "arc",      f"{CACHE_DIR}/arc_cache.json")

TASK_CACHE = {"mnli": mnli_cache, "bbh_date": bbh_cache, "arc": arc_cache}

for task, cache in TASK_CACHE.items():
    n_perts = len(list(cache.values())[0])
    pert_names = sorted(list(cache.values())[0].keys())
    print(f"  {task}: {len(cache)} examples x {n_perts} perturbations")
    print(f"    Names: {pert_names}")

# Consistency check
e1 = get_cached_entry(mnli_cache, 0, "lex_rewrite")
e2 = get_cached_entry(mnli_cache, 0, "lex_rewrite")
print(f"\nConsistency check: {e1 == e2} (must be True)")
print(f"Entry keys: {list(e1.keys())} (must include prompt and instruction)")

Building/loading caches...
NOTE: cache files are now self-invalidating via fingerprint checks.

  Materializing current cache for mnli (300 examples)...
    50/300 done
    100/300 done
    150/300 done
    200/300 done
    250/300 done
    300/300 done
  No existing cache file: perturbation_cache/mnli_cache.json
  Cache saved: perturbation_cache/mnli_cache.json
  Materializing current cache for bbh_date (250 examples)...
    50/250 done
    100/250 done
    150/250 done
    200/250 done
    250/250 done
  No existing cache file: perturbation_cache/bbh_cache.json
  Cache saved: perturbation_cache/bbh_cache.json
  Materializing current cache for arc (300 examples)...
    50/300 done
    100/300 done
    150/300 done
    200/300 done
    250/300 done
    300/300 done
  No existing cache file: perturbation_cache/arc_cache.json
  Cache saved: perturbation_cache/arc_cache.json
  mnli: 300 examples x 10 perturbations
    Names: ['baseline', 'ctx_irrelevant', 'ctx_role', 'fmt_caps', 'fmt_sepa

## Cell 11 - Semantic Embeddings and Sensitivity Score

[v6] sensitivity_score = flip_rate × |acc_drop|
- This is a SENSITIVITY metric, not a fragility metric: abs(acc_drop) means both
  improvements and degradations contribute. Always pair with direction and signed acc_drop
  when interpreting.
- Do not rank "most fragile" by SS alone — use signed acc_drop for that.

[v6.1] semantic_distance renamed to instruction_semantic_distance in documentation.
The metric compares instruction strings only (not full prompts). For perturbations like
struct_inst_first and fmt_separators, the instruction is unchanged so d_sem = 0 even
though the full prompt differs. This is correct but must be stated clearly in the paper.


In [20]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

sem_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


def instruction_semantic_distance(text_a: str, text_b: str) -> float:
    """
    Cosine distance between sentence embeddings of two instruction strings.

    Named instruction_semantic_distance (not just semantic_distance) because it
    operates on instruction templates only, not full prompts. For perturbations
    that change prompt structure without changing instruction wording (e.g.
    struct_inst_first, fmt_separators), this will be 0 — which is correct:
    the instruction itself did not change semantically.
    """
    emb_a = sem_model.encode(text_a, convert_to_numpy=True)
    emb_b = sem_model.encode(text_b, convert_to_numpy=True)
    return float(cosine(emb_a, emb_b))


# Backward-compatible alias
semantic_distance = instruction_semantic_distance


def sensitivity_score(flip_rate: float, acc_drop: float) -> float:
    """
    Bounded sensitivity metric: flip_rate × |acc_drop|

    KNOWN LIMITATION: If errors are symmetric (equal flips correct→wrong and
    wrong→correct), acc_drop = 0 and SS = 0 even though the model was highly
    unstable. Always report flip_rate alongside SS to catch this case.
    A high flip_rate with SS ≈ 0 indicates symmetric instability.

    NOTE: This measures sensitivity (instability in EITHER direction), not fragility.
    abs(acc_drop) means a perturbation that improves accuracy also produces a non-zero
    score. When reporting:
    - Always pair SS with flip_rate (catches symmetric instability)
    - Pair with direction and signed acc_drop
    - Do NOT rank "most fragile perturbations" by SS alone
    - For fragility ranking, use signed acc_drop filtered to direction="drop"
    """
    return flip_rate * abs(acc_drop)


# Backward-compatible alias
def fragility_index(flip_rate: float, acc_drop: float, d_sem: float = 0.0) -> float:
    """Deprecated wrapper — calls sensitivity_score, ignores d_sem."""
    return sensitivity_score(flip_rate, acc_drop)


d = instruction_semantic_distance(
    "Answer with one letter only: A for entailment, B for neutral, C for contradiction.",
    "Determine the relation between the premise and the hypothesis.\n"
    "Answer with one letter only: A for entailment, B for neutral, C for contradiction.",
)
print(f"Instruction semantic distance (baseline vs rewrite): {d:.4f}")
print(f"Sensitivity score example: FR=0.3, dacc=5.0 -> {sensitivity_score(0.3, 5.0):.2f}")
print(f"  (Note: SS uses abs(acc_drop), so improvements also score > 0)")
print("Semantic pipeline ready.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Instruction semantic distance (baseline vs rewrite): 0.2292
Sensitivity score example: FR=0.3, dacc=5.0 -> 1.50
  (Note: SS uses abs(acc_drop), so improvements also score > 0)
Semantic pipeline ready.


## Cell 11a - Single-Token Label Verification

[REVIEW Fix 5] Verifies that every candidate label for every task encodes to exactly
one token for the current model. If any label fails, scoring is wrong for that label.
Call this after every load_model().

[v7] Fix C1+C2: verify_single_token_labels now uses continuation_token_ids() —
the same helper used by get_label_logprobs — ensuring verifier and scorer agree.


In [21]:
def verify_single_token_labels(model_key: str, tokenizer) -> bool:
    """
    Preflight check for label scoring.

    Instruct models:
        build a real probe prompt and verify each label is exactly one
        continuation token under the same strict path used by the scorer.

    Base models:
        verify each label has at least one acceptable immediate 1-token variant
        among {label, " " + label, "\\n" + label}, matching the base scorer.
    """
    all_labels = set()
    all_labels.update(["A", "B", "C"])  # MNLI
    for ex in bbh_data:
        all_labels.update(ex["candidate_labels"])
    for ex in arc_data:
        all_labels.update(ex["candidate_labels"])

    info = MODEL_REGISTRY[model_key]

    if info["chat_template"] is None:
        failed = []
        for label in sorted(all_labels):
            variant_ids = label_variant_token_ids(tokenizer, label)
            if not variant_ids:
                failed.append(label)

        if failed:
            msg = (
                f"{model_key}: labels with no acceptable 1-token immediate variant "
                f"among {{label, ' '+label, '\\n'+label}}:\n"
            )
            for label in failed:
                msg += f"  {label!r}\n"
            raise RuntimeError(msg)

        print(
            f"Base-label preflight passed for {model_key}: "
            f"all {len(all_labels)} labels have at least one 1-token immediate variant."
        )
        return True

    # Instruct model: use the exact strict continuation path on a real probe prompt.
    probe_msg = "Choose A or B.\nAnswer:"
    probe_ids, probe_str = format_and_tokenize(model_key, probe_msg, tokenizer=tokenizer)
    probe_ids = _extract_input_ids_tensor(probe_ids)[0].tolist()

    failed = []
    for label in sorted(all_labels):
        try:
            cont_ids = continuation_token_ids(
                tokenizer,
                label,
                prompt_ids=probe_ids,
                prompt_str=probe_str,
            )
            if len(cont_ids) != 1:
                failed.append((label, cont_ids))
        except Exception as e:
            failed.append((label, str(e)))

    if failed:
        msg = f"{model_key}: labels failing strict instruct continuation check:\n"
        for label, detail in failed:
            msg += f"  {label!r} -> {detail}\n"
        raise RuntimeError(msg)

    print(
        f"Strict instruct-label check passed for {model_key}: "
        f"all {len(all_labels)} labels are exactly 1 continuation token in the probe prompt."
    )
    return True


print("verify_single_token_labels() defined.")

verify_single_token_labels() defined.


## Cell 12 - Evaluation Loop

[VISHNU] candidate_labels read per example.
[MUDIT] prompt read from cache via get_cached_entry().
[REVIEW Fix 17] Stores margin (confidence) alongside each prediction for paired CSV export.

In [22]:
def evaluate_dataset(
    model, tokenizer, model_key: str,
    data: List[dict], pert_name: str, task: str, cache: dict,
    batch_desc: str = "", max_examples: int = None,
) -> List[dict]:
    """
    [VISHNU] candidate_labels per example.
    [MUDIT] prompt from cache — same string used in eval and d_sem.
    [REVIEW Fix 17] Returns margin field for paired CSV export.
    """
    if max_examples:
        data = data[:max_examples]

    results = []
    correct_count = 0
    desc = batch_desc or f"{model_key}/{task}/{pert_name}"

    for i, ex in enumerate(data):
        entry = get_cached_entry(cache, i, pert_name)
        prompt = entry["prompt"]

        if ex["task"] == "mnli":
            candidate_labels = ["A", "B", "C"]
        elif "candidate_labels" in ex:
            candidate_labels = ex["candidate_labels"]
        else:
            raise ValueError(f"No candidate_labels on example {i}, task {ex['task']}")

        result = get_label_logprobs(
            model, tokenizer, model_key,
            prompt, candidate_labels, ex["gold_label"]
        )
        results.append({
            "idx":             i,
            "prompt":          prompt,
            "gold_label":      result.gold_label,
            "predicted_label": result.predicted_label,
            "correct":         result.correct,
            "label_logprobs":  result.label_logprobs,
            "margin":          result.margin,
        })
        if result.correct:
            correct_count += 1
        if (i + 1) % 50 == 0:
            acc = correct_count / (i + 1) * 100
            print(f"  [{desc}] {i+1}/{len(data)} acc: {acc:.1f}%")

    final_acc = correct_count / len(data) * 100
    print(f"  [{desc}] Final: {final_acc:.1f}% ({correct_count}/{len(data)})")
    return results


def compute_metrics(baseline_results: List[dict], perturbed_results: List[dict]) -> dict:
    n = len(baseline_results)
    assert n == len(perturbed_results)
    base_acc = sum(r["correct"] for r in baseline_results) / n
    pert_acc = sum(r["correct"] for r in perturbed_results) / n
    acc_drop = (base_acc - pert_acc) * 100
    flips = sum(
        1 for b, p in zip(baseline_results, perturbed_results)
        if b["predicted_label"] != p["predicted_label"]
    )
    flip_rate = flips / n
    return {
        "base_accuracy": round(base_acc * 100, 2),
        "pert_accuracy": round(pert_acc * 100, 2),
        "accuracy_drop": round(acc_drop, 2),
        "flip_rate":     round(flip_rate, 4),
        "consistency":   round(1.0 - flip_rate, 4),
        "n_examples":    n,
        # [v7 Fix P2] Raw values for sensitivity_score computation (avoid rounding before multiply)
        "_raw_flip_rate": flip_rate,
        "_raw_acc_drop":  acc_drop,
    }


print("Evaluation functions ready.")

Evaluation functions ready.


## Cell 13 - Experiment Runner

[v6] Changes:
- ALL_PERT_NAMES updated: lex_synonym→lex_rewrite, struct_inst_last removed,
  para_rewrite→instruction_rewrite
- sensitivity_score replaces fragility_index (no d_sem in denominator)
- d_sem still computed and stored as separate metric
- Collapse diagnostics added: n_unique_preds, majority_label, majority_frac
- Perturbation audit CSV exported after cache build


In [23]:
TASK_CONFIG = {
    "mnli":     {"data": None},
    "bbh_date": {"data": None},
    "arc":      {"data": None},
}

# ALL_PERT_NAMES and PERT_CATEGORIES defined in Cell 4 (config)


def write_paired_csv(model_key, task, pert_name, baseline_results, pert_results):
    """Write per-example paired CSV with flip and margin columns."""
    path = f"results/paired/{model_key}_{task}_{pert_name}.csv"
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "idx", "gold", "base_pred", "pert_pred",
            "base_correct", "pert_correct", "flip",
            "base_margin", "pert_margin",
        ])
        writer.writeheader()
        for b, p in zip(baseline_results, pert_results):
            writer.writerow({
                "idx":          b["idx"],
                "gold":         b["gold_label"],
                "base_pred":    b["predicted_label"],
                "pert_pred":    p["predicted_label"],
                "base_correct": int(b["correct"]),
                "pert_correct": int(p["correct"]),
                "flip":         int(b["predicted_label"] != p["predicted_label"]),
                "base_margin":  round(b["margin"], 4),
                "pert_margin":  round(p["margin"], 4),
            })


def collapse_diagnostics(results: list) -> dict:
    """
    [v6] Detect prediction collapse: when a model is stuck on one label.
    """
    from collections import Counter
    preds = [r["predicted_label"] for r in results]
    counts = Counter(preds)
    n = len(preds)
    majority_label, majority_count = counts.most_common(1)[0]
    return {
        "n_unique_preds":   len(counts),
        "majority_label":   majority_label,
        "majority_frac":    round(majority_count / n, 4),
        "collapsed":        majority_count / n > 0.90 or len(counts) == 1,
    }


def export_perturbation_audit(cache_dict: dict, task: str, save_dir: str = "results"):
    """
    [v7 Fix Re1] Export a CSV showing what each perturbation did.
    Now includes pairwise d_sem between all perturbation pairs to surface
    inter-perturbation redundancy (e.g. lex_rewrite vs instruction_rewrite).

    NOTE: Preview text and pairwise d_sem use example 0 only. For lex_typo,
    the actual typo pattern varies per example (d_sem is averaged over 30
    examples in the experiment runner). The audit CSV shows one representative
    sample — not the aggregate.
    """
    rows = []
    entry = cache_dict[0]
    base_instr = entry["baseline"]["instruction"]
    base_prompt = entry["baseline"]["prompt"]

    # Collect all perturbation instructions for pairwise comparison
    pert_instrs = {}
    for pert_name, pert_entry in entry.items():
        if pert_name == "baseline":
            continue
        pert_instrs[pert_name] = pert_entry["instruction"]

    for pert_name, pert_entry in entry.items():
        if pert_name == "baseline":
            continue
        pert_instr = pert_entry["instruction"]
        pert_prompt = pert_entry["prompt"]
        d_sem = instruction_semantic_distance(base_instr, pert_instr)
        cat = PERT_CATEGORIES.get(pert_name, "unknown")
        instr_changed = base_instr != pert_instr
        prompt_changed = base_prompt != pert_prompt

        # [v7 Re1] Find closest other perturbation (pairwise redundancy check)
        min_cross_dsem = float("inf")
        closest_pert = ""
        for other_name, other_instr in pert_instrs.items():
            if other_name == pert_name:
                continue
            cross_d = instruction_semantic_distance(pert_instr, other_instr)
            if cross_d < min_cross_dsem:
                min_cross_dsem = cross_d
                closest_pert = other_name

        rows.append({
            "task": task,
            "perturbation": pert_name,
            "category": cat,
            "base_instruction": base_instr,
            "perturbed_instruction": pert_instr,
            "instruction_changed": instr_changed,
            "prompt_changed": prompt_changed,
            "base_prompt_preview": base_prompt[:150],
            "perturbed_prompt_preview": pert_prompt[:150],
            "instruction_semantic_distance": round(d_sem, 6),
            "closest_other_pert": closest_pert,
            "min_pairwise_dsem": round(min_cross_dsem, 6) if min_cross_dsem < float("inf") else None,
        })
    return rows


def run_full_experiment(
    model, tokenizer, model_key: str,
    n_mnli: int = N_MNLI, n_bbh: int = N_BBH, n_arc: int = N_ARC,
    save_dir: str = "results",
) -> dict:
    os.makedirs(f"{save_dir}/{model_key}", exist_ok=True)
    os.makedirs(f"{save_dir}/paired", exist_ok=True)
    TASK_CONFIG["mnli"]["data"]     = mnli_data[:n_mnli]
    TASK_CONFIG["bbh_date"]["data"] = bbh_data[:n_bbh]
    TASK_CONFIG["arc"]["data"]      = arc_data[:n_arc]
    all_results = {}

    for task, cfg in TASK_CONFIG.items():
        print(f"\n{'='*60}")
        print(f"Task: {task.upper()} | Model: {model_key}")
        print(f"{'='*60}")
        all_results[task] = {}
        cache = TASK_CACHE[task]

        baseline_results = evaluate_dataset(
            model, tokenizer, model_key,
            cfg["data"], "baseline", task, cache,
            batch_desc=f"{task}/baseline",
        )
        base_diag = collapse_diagnostics(baseline_results)
        base_metrics = compute_metrics(baseline_results, baseline_results)
        base_metrics.update(base_diag)
        all_results[task]["baseline"] = {
            "results": baseline_results,
            "metrics": base_metrics,
        }
        if base_diag["collapsed"]:
            print(f"  ⚠ BASELINE COLLAPSE: {base_diag['majority_label']} "
                  f"= {base_diag['majority_frac']*100:.1f}% of predictions")

        base_path = f"{save_dir}/{model_key}/{task}_baseline.json"
        with open(base_path, "w") as f:
            json.dump(baseline_results, f, indent=2)

        for pert_name in ALL_PERT_NAMES:
            if pert_name == "baseline":
                continue
            category = PERT_CATEGORIES[pert_name]
            print(f"\n  [{category}] {pert_name}")

            pert_results = evaluate_dataset(
                model, tokenizer, model_key,
                cfg["data"], pert_name, task, cache,
                batch_desc=f"{task}/{pert_name}",
            )
            metrics = compute_metrics(baseline_results, pert_results)

            # [v7 Fix Mo2] instruction-level semantic distance averaged over
            # a sample of examples (not just example 0). This matters for
            # lex_typo where each example gets a different random substitution.
            _dsem_sample_n = min(30, len(cfg["data"]))
            _dsem_vals = []
            for _si in range(_dsem_sample_n):
                _base_instr = get_cached_entry(cache, _si, "baseline")["instruction"]
                _pert_instr = get_cached_entry(cache, _si, pert_name)["instruction"]
                _dsem_vals.append(instruction_semantic_distance(_base_instr, _pert_instr))
            d_sem = sum(_dsem_vals) / len(_dsem_vals)

            # [v7 Fix P2] Use raw floats, not pre-rounded values
            ss = sensitivity_score(metrics["_raw_flip_rate"], metrics["_raw_acc_drop"])
            pert_diag = collapse_diagnostics(pert_results)

            metrics["semantic_distance"]  = round(d_sem, 6)
            metrics["sensitivity_score"]  = round(ss, 4)
            metrics["category"]           = category
            metrics.update(pert_diag)

            if metrics["accuracy_drop"] > 0:
                metrics["direction"] = "drop"
            elif metrics["accuracy_drop"] < 0:
                metrics["direction"] = "improvement"
            else:
                metrics["direction"] = "no_change"

            all_results[task][pert_name] = {"metrics": metrics}

            pert_path = f"{save_dir}/{model_key}/{task}_{pert_name}.json"
            with open(pert_path, "w") as f:
                json.dump({"metrics": metrics, "pert_results": pert_results}, f, indent=2)

            write_paired_csv(model_key, task, pert_name, baseline_results, pert_results)

            collapse_flag = " ⚠ COLLAPSED" if pert_diag["collapsed"] else ""
            print(f"     delta_acc={metrics['accuracy_drop']:+.1f}pp "
                  f"FR={metrics['flip_rate']:.3f} "
                  f"SS={metrics['sensitivity_score']:.2f} "
                  f"d_sem={d_sem:.4f}{collapse_flag}")

    summary = {
        task: {pn: v["metrics"] for pn, v in perts.items()}
        for task, perts in all_results.items()
    }
    with open(f"{save_dir}/{model_key}/SUMMARY.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nDone. Results in {save_dir}/{model_key}/")
    return all_results


print("Experiment runner ready.")


Experiment runner ready.


## Cell 14 - Sanity Check

[REVIEW Fix 5] verify_single_token_labels() called after loading model.

In [24]:
model, tokenizer = load_model("qwen2.5-7b-instruct")

# [REVIEW Fix 5] Always run this after loading a model
verify_single_token_labels("qwen2.5-7b-instruct", tokenizer)

# MNLI gold labels are A/B/C
print("\nMNLI gold label check:")
print(f"  Sample: {[ex['gold_label'] for ex in mnli_data[:6]]}")
print(f"  All valid: {all(ex['gold_label'] in ['A','B','C'] for ex in mnli_data)}")

# MNLI logprob scoring
test_ex = mnli_data[0]
result = get_label_logprobs(
    model, tokenizer, "qwen2.5-7b-instruct",
    make_mnli_prompt(test_ex), ["A", "B", "C"], test_ex["gold_label"]
)
print(f"\nMNLI inference: gold={test_ex['gold_label']} pred={result.predicted_label} "
      f"correct={result.correct} margin={result.margin:.3f}")

# BBH and ARC dynamic labels
print("\nBBH label samples:")
for i in range(3):
    print(f"  ex {i}: labels={bbh_data[i]['candidate_labels']} gold={bbh_data[i]['gold_label']}")

print("\nARC label samples:")
for i in range(3):
    print(f"  ex {i}: labels={arc_data[i]['candidate_labels']} gold={arc_data[i]['gold_label']}")

# Cache format check
entry = get_cached_entry(mnli_cache, 0, "lex_rewrite")
print(f"\nCache entry keys: {list(entry.keys())} (must have prompt and instruction)")
print(f"  prompt[:60]:      {entry['prompt'][:60]}")
print(f"  instruction[:60]: {entry['instruction'][:60]}")

# d_sem from instruction strings
base_instr = get_cached_entry(mnli_cache, 0, "baseline")["instruction"]
pert_instr = get_cached_entry(mnli_cache, 0, "instruction_rewrite")["instruction"]
d = semantic_distance(base_instr, pert_instr)
print(f"\nInstruction-level d_sem (baseline vs instruction_rewrite): {d:.4f}")

# Sensitivity score is bounded
print(f"Sensitivity score: FR=0.5, dacc=20 -> {sensitivity_score(0.5, 20):.2f}")

# [v6.1] format_and_tokenize uses tokenize=True, format_prompt for display only
test_ids, test_prompt = format_and_tokenize("qwen2.5-7b-instruct", "test", tokenizer=tokenizer)
print(f"\nChat template output preview: {test_prompt[:80]}...")
print(f"Token IDs shape: {test_ids.shape}")
print(f"Qwen tags present: {'<|im_start|>' in test_prompt}")

# Lowercase fix restores answer letters
low = perturb_formality_lower("Answer with one letter only: A for entailment, B for neutral.")
print(f"\nLowercase fix: {low}")
print(f"  A/B preserved: {'A' in low and 'B' in low} (must be True)")

# [v6] lex_rewrite preserves labels
lex_test = perturb_lexical(
    "Answer with one letter only: A for entailment, B for neutral, C for contradiction.", "mnli"
)
print(f"\nLexical rewrite preserves labels: {'A for entailment' in lex_test and 'C for contradiction' in lex_test}")

# Body isolation check
body_test, instr_test = mnli_parts(mnli_data[0])
lex_result = join_prompt(body_test, perturb_lexical(instr_test, "mnli"))
assert body_test in lex_result, "Body changed after lexical perturbation"
print(f"Body isolation check: body unchanged after lex_rewrite = True")

print("\nAll checks done.")


No model loaded.
Loading qwen2.5-7b-instruct from Qwen/Qwen2.5-7B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded in 63s, VRAM used: 2.3 GB
Strict instruct-label check passed for qwen2.5-7b-instruct: all 6 labels are exactly 1 continuation token in the probe prompt.

MNLI gold label check:
  Sample: ['A', 'A', 'B', 'C', 'C', 'C']
  All valid: True

MNLI inference: gold=A pred=A correct=True margin=7.093

BBH label samples:
  ex 0: labels=['A', 'B', 'C', 'D', 'E', 'F'] gold=B
  ex 1: labels=['A', 'B', 'C', 'D', 'E', 'F'] gold=A
  ex 2: labels=['A', 'B', 'C', 'D', 'E', 'F'] gold=B

ARC label samples:
  ex 0: labels=['A', 'B', 'C', 'D'] gold=B
  ex 1: labels=['A', 'B', 'C', 'D'] gold=C
  ex 2: labels=['A', 'B', 'C', 'D'] gold=D

Cache entry keys: ['prompt', 'instruction'] (must have prompt and instruction)
  prompt[:60]:      Premise: I touched my palm to his mutilated cheek, and tried
  instruction[:60]: What is the relationship between the premise and the hypothe

Instruction-level d_sem (baseline vs instruction_rewrite): 0.1482
Sensitivity score: FR=0.5, dacc=20 -> 10.00

Chat template outp

## Cell 15 - Run Experiments: Llama-3.1-8B-Instruct

In [ ]:
model, tokenizer = load_model("llama3.1-8b-instruct")
verify_single_token_labels("llama3.1-8b-instruct", tokenizer)  # [REVIEW Fix 5]
llama_results = run_full_experiment(model, tokenizer, "llama3.1-8b-instruct")
clear_model()

Clearing qwen2.5-7b-instruct from memory.
GPU freed. Available: 82.8 GB
Loading llama3.1-8b-instruct from meta-llama/Meta-Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded in 68s, VRAM used: 3.8 GB
Strict instruct-label check passed for llama3.1-8b-instruct: all 6 labels are exactly 1 continuation token in the probe prompt.

Task: MNLI | Model: llama3.1-8b-instruct
  [mnli/baseline] 50/300 acc: 32.0%
  [mnli/baseline] 100/300 acc: 46.0%
  [mnli/baseline] 150/300 acc: 46.0%
  [mnli/baseline] 200/300 acc: 45.0%
  [mnli/baseline] 250/300 acc: 46.0%
  [mnli/baseline] 300/300 acc: 44.3%
  [mnli/baseline] Final: 44.3% (133/300)

  [lexical] lex_rewrite
  [mnli/lex_rewrite] 50/300 acc: 68.0%
  [mnli/lex_rewrite] 100/300 acc: 69.0%
  [mnli/lex_rewrite] 150/300 acc: 66.7%
  [mnli/lex_rewrite] 200/300 acc: 67.0%
  [mnli/lex_rewrite] 250/300 acc: 64.4%
  [mnli/lex_rewrite] 300/300 acc: 63.3%
  [mnli/lex_rewrite] Final: 63.3% (190/300)
     delta_acc=-19.0pp FR=0.403 SS=7.66 d_sem=0.1114

  [lexical] lex_typo
  [mnli/lex_typo] 50/300 acc: 34.0%
  [mnli/lex_typo] 100/300 acc: 44.0%
  [mnli/lex_typo] 150/300 acc: 43.3%
  [mnli/lex_typo] 200/300 acc: 41.0%
  [mn

## Cell 16 - Run Experiments: Gemma-2-9B-IT

In [26]:
model, tokenizer = load_model("gemma2-9b-it")
verify_single_token_labels("gemma2-9b-it", tokenizer)  # [REVIEW Fix 5]
gemma_results = run_full_experiment(model, tokenizer, "gemma2-9b-it")
clear_model()

  [mnli/fmt_caps] 150/300 acc: 51.3%
  [mnli/fmt_caps] 200/300 acc: 54.0%
  [mnli/fmt_caps] 250/300 acc: 54.8%
  [mnli/fmt_caps] 300/300 acc: 55.0%
  [mnli/fmt_caps] Final: 55.0% (165/300)
     delta_acc=+3.3pp FR=0.130 SS=0.43 d_sem=0.0000

  [formatting] fmt_separators
  [mnli/fmt_separators] 50/300 acc: 66.0%
  [mnli/fmt_separators] 100/300 acc: 56.0%
  [mnli/fmt_separators] 150/300 acc: 56.0%
  [mnli/fmt_separators] 200/300 acc: 59.0%
  [mnli/fmt_separators] 250/300 acc: 58.4%
  [mnli/fmt_separators] 300/300 acc: 57.7%
  [mnli/fmt_separators] Final: 57.7% (173/300)
     delta_acc=+0.7pp FR=0.047 SS=0.03 d_sem=0.0000

  [rewrite] instruction_rewrite
  [mnli/instruction_rewrite] 50/300 acc: 60.0%
  [mnli/instruction_rewrite] 100/300 acc: 62.0%
  [mnli/instruction_rewrite] 150/300 acc: 61.3%
  [mnli/instruction_rewrite] 200/300 acc: 61.0%
  [mnli/instruction_rewrite] 250/300 acc: 60.8%
  [mnli/instruction_rewrite] 300/300 acc: 61.7%
  [mnli/instruction_rewrite] Final: 61.7% (185/300)


## Cell 17 - Run Experiments: Qwen2.5-7B-Instruct

In [27]:
model, tokenizer = load_model("qwen2.5-7b-instruct")
verify_single_token_labels("qwen2.5-7b-instruct", tokenizer)  # [REVIEW Fix 5]
qwen_results = run_full_experiment(model, tokenizer, "qwen2.5-7b-instruct")
clear_model()

No model loaded.
Loading qwen2.5-7b-instruct from Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded in 65s, VRAM used: 6.5 GB
Strict instruct-label check passed for qwen2.5-7b-instruct: all 6 labels are exactly 1 continuation token in the probe prompt.

Task: MNLI | Model: qwen2.5-7b-instruct
  [mnli/baseline] 50/300 acc: 82.0%
  [mnli/baseline] 100/300 acc: 81.0%
  [mnli/baseline] 150/300 acc: 78.0%
  [mnli/baseline] 200/300 acc: 79.5%
  [mnli/baseline] 250/300 acc: 78.8%
  [mnli/baseline] 300/300 acc: 80.3%
  [mnli/baseline] Final: 80.3% (241/300)

  [lexical] lex_rewrite
  [mnli/lex_rewrite] 50/300 acc: 86.0%
  [mnli/lex_rewrite] 100/300 acc: 86.0%
  [mnli/lex_rewrite] 150/300 acc: 84.7%
  [mnli/lex_rewrite] 200/300 acc: 83.5%
  [mnli/lex_rewrite] 250/300 acc: 83.6%
  [mnli/lex_rewrite] 300/300 acc: 84.7%
  [mnli/lex_rewrite] Final: 84.7% (254/300)
     delta_acc=-4.3pp FR=0.100 SS=0.43 d_sem=0.1114

  [lexical] lex_typo
  [mnli/lex_typo] 50/300 acc: 84.0%
  [mnli/lex_typo] 100/300 acc: 86.0%
  [mnli/lex_typo] 150/300 acc: 82.7%
  [mnli/lex_typo] 200/300 acc: 82.0%
  [mnli/

## Cell 18 - Run Base Models

[NEHA] All three base models. Gemma-2-9B base is new — previously only Llama
and Qwen had base variants, making the instruct vs base comparison asymmetric.

In [28]:
model, tokenizer = load_model("llama3.1-8b-base")
verify_single_token_labels("llama3.1-8b-base", tokenizer)
llama_base_results = run_full_experiment(model, tokenizer, "llama3.1-8b-base")
clear_model()

model, tokenizer = load_model("gemma2-9b-base")  # [NEHA] new
verify_single_token_labels("gemma2-9b-base", tokenizer)
gemma_base_results = run_full_experiment(model, tokenizer, "gemma2-9b-base")
clear_model()

model, tokenizer = load_model("qwen2.5-7b-base")
verify_single_token_labels("qwen2.5-7b-base", tokenizer)
qwen_base_results = run_full_experiment(model, tokenizer, "qwen2.5-7b-base")
clear_model()

print("Base model experiments complete.")

No model loaded.
Loading llama3.1-8b-base from meta-llama/Meta-Llama-3.1-8B


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded in 183s, VRAM used: 4.6 GB
Base-label preflight passed for llama3.1-8b-base: all 6 labels have at least one 1-token immediate variant.

Task: MNLI | Model: llama3.1-8b-base
  [mnli/baseline] 50/300 acc: 22.0%
  [mnli/baseline] 100/300 acc: 35.0%
  [mnli/baseline] 150/300 acc: 34.7%
  [mnli/baseline] 200/300 acc: 33.5%
  [mnli/baseline] 250/300 acc: 34.0%
  [mnli/baseline] 300/300 acc: 33.3%
  [mnli/baseline] Final: 33.3% (100/300)
  ⚠ BASELINE COLLAPSE: A = 100.0% of predictions

  [lexical] lex_rewrite
  [mnli/lex_rewrite] 50/300 acc: 22.0%
  [mnli/lex_rewrite] 100/300 acc: 35.0%
  [mnli/lex_rewrite] 150/300 acc: 34.7%
  [mnli/lex_rewrite] 200/300 acc: 33.5%
  [mnli/lex_rewrite] 250/300 acc: 34.0%
  [mnli/lex_rewrite] 300/300 acc: 33.3%
  [mnli/lex_rewrite] Final: 33.3% (100/300)
     delta_acc=+0.0pp FR=0.000 SS=0.00 d_sem=0.1114 ⚠ COLLAPSED

  [lexical] lex_typo
  [mnli/lex_typo] 50/300 acc: 22.0%
  [mnli/lex_typo] 100/300 acc: 35.0%
  [mnli/lex_typo] 150/300 acc: 33.3%
  [mn

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Loaded in 276s, VRAM used: 5.4 GB
Base-label preflight passed for gemma2-9b-base: all 6 labels have at least one 1-token immediate variant.

Task: MNLI | Model: gemma2-9b-base
  [mnli/baseline] 50/300 acc: 22.0%
  [mnli/baseline] 100/300 acc: 35.0%
  [mnli/baseline] 150/300 acc: 34.7%
  [mnli/baseline] 200/300 acc: 33.5%
  [mnli/baseline] 250/300 acc: 34.0%
  [mnli/baseline] 300/300 acc: 33.3%
  [mnli/baseline] Final: 33.3% (100/300)
  ⚠ BASELINE COLLAPSE: A = 100.0% of predictions

  [lexical] lex_rewrite
  [mnli/lex_rewrite] 50/300 acc: 22.0%
  [mnli/lex_rewrite] 100/300 acc: 35.0%
  [mnli/lex_rewrite] 150/300 acc: 34.7%
  [mnli/lex_rewrite] 200/300 acc: 33.5%
  [mnli/lex_rewrite] 250/300 acc: 34.0%
  [mnli/lex_rewrite] 300/300 acc: 33.3%
  [mnli/lex_rewrite] Final: 33.3% (100/300)
     delta_acc=+0.0pp FR=0.000 SS=0.00 d_sem=0.1114 ⚠ COLLAPSED

  [lexical] lex_typo
  [mnli/lex_typo] 50/300 acc: 32.0%
  [mnli/lex_typo] 100/300 acc: 38.0%
  [mnli/lex_typo] 150/300 acc: 37.3%
  [mnli/l

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded in 149s, VRAM used: 6.5 GB
Base-label preflight passed for qwen2.5-7b-base: all 6 labels have at least one 1-token immediate variant.

Task: MNLI | Model: qwen2.5-7b-base
  [mnli/baseline] 50/300 acc: 32.0%
  [mnli/baseline] 100/300 acc: 41.0%
  [mnli/baseline] 150/300 acc: 42.7%
  [mnli/baseline] 200/300 acc: 41.5%
  [mnli/baseline] 250/300 acc: 41.2%
  [mnli/baseline] 300/300 acc: 40.7%
  [mnli/baseline] Final: 40.7% (122/300)

  [lexical] lex_rewrite
  [mnli/lex_rewrite] 50/300 acc: 58.0%
  [mnli/lex_rewrite] 100/300 acc: 61.0%
  [mnli/lex_rewrite] 150/300 acc: 59.3%
  [mnli/lex_rewrite] 200/300 acc: 59.0%
  [mnli/lex_rewrite] 250/300 acc: 59.2%
  [mnli/lex_rewrite] 300/300 acc: 59.7%
  [mnli/lex_rewrite] Final: 59.7% (179/300)
     delta_acc=-19.0pp FR=0.307 SS=5.83 d_sem=0.1114

  [lexical] lex_typo
  [mnli/lex_typo] 50/300 acc: 50.0%
  [mnli/lex_typo] 100/300 acc: 62.0%
  [mnli/lex_typo] 150/300 acc: 58.0%
  [mnli/lex_typo] 200/300 acc: 60.0%
  [mnli/lex_typo] 250/300 acc:

## Cell 19 - Results Summary

[v6] FI replaced with sensitivity_score (SS = flip_rate × |acc_drop|).
Category averages now use avg_SS instead of avg_FI.
struct_inst_last removed. lex_synonym → lex_rewrite. para_rewrite → instruction_rewrite.
Collapse diagnostics shown for flagged runs.


In [29]:
def load_summary(model_key: str, save_dir: str = "results") -> dict:
    path = f"{save_dir}/{model_key}/SUMMARY.json"
    if not os.path.exists(path):
        print(f"No summary found for {model_key}")
        return {}
    with open(path) as f:
        return json.load(f)


INSTRUCT_MODELS = [
    "llama3.1-8b-instruct",
    "gemma2-9b-it",
    "qwen2.5-7b-instruct",
]

BASE_MODELS = [
    "llama3.1-8b-base",
    "gemma2-9b-base",
    "qwen2.5-7b-base",
]

# [v6] Display with sensitivity_score instead of FI
for task in ["mnli", "bbh_date", "arc"]:
    print(f"\n{'='*110}")
    print(f"Task: {task.upper()} — Instruct models")
    header = f"  {'Perturbation':<22} {'Cat':<12}"
    for m in INSTRUCT_MODELS:
        s = m.split("-")[0].upper()
        header += f"  {s+' acc':>8} {s+' dacc':>8} {s+' FR':>6} {s+' SS':>8} {s+' d_sem':>7}"
    print(header)
    print("-"*110)

    summaries = {m: load_summary(m) for m in INSTRUCT_MODELS}
    all_perts = set()
    for s in summaries.values():
        if task in s:
            all_perts.update(s[task].keys())
    all_perts.discard("baseline")

    for pert in sorted(all_perts):
        row_cat = ""
        line = f"  {pert:<22} "
        for m in INSTRUCT_MODELS:
            met = summaries.get(m, {}).get(task, {}).get(pert, {})
            if met:
                row_cat = met.get("category", "")[:11]
                collapsed = " !" if met.get("collapsed", False) else ""
                line += f"  {met.get('pert_accuracy',0):>8.1f} "
                line += f"{met.get('accuracy_drop',0):>+8.1f} "
                line += f"{met.get('flip_rate',0):>6.3f} "
                line += f"{met.get('sensitivity_score',0):>8.2f} "
                line += f"{met.get('semantic_distance',0):>7.4f}{collapsed}"
            else:
                line += f"  {'N/A':>8} {'N/A':>8} {'N/A':>6} {'N/A':>8} {'N/A':>7}"
        print(line + f"  [{row_cat}]")

print("\n" + "="*110)
print("\nAverage Sensitivity Score by category — per model")
for task in ["mnli", "bbh_date", "arc"]:
    print(f"\n  {task.upper()}")
    header = f"    {'Category':<14}"
    for m in INSTRUCT_MODELS:
        s = m.split('-')[0].upper()
        header += f" {s+' dacc':>10} {s+' FR':>8} {s+' SS':>8} {s+' d_sem':>8}"
    print(header)
    print(f"    {'-'*100}")
    summaries = {m: load_summary(m) for m in INSTRUCT_MODELS}
    all_cats = set()
    for s in summaries.values():
        for p, met in s.get(task, {}).items():
            if p != "baseline" and "category" in met:
                all_cats.add(met["category"])
    for cat in sorted(all_cats):
        line = f"    {cat:<14}"
        for m in INSTRUCT_MODELS:
            s = summaries.get(m, {})
            dacc_vals, fr_vals, ss_vals, dsem_vals = [], [], [], []
            for p, met in s.get(task, {}).items():
                if p != "baseline" and met.get("category") == cat:
                    dacc_vals.append(met.get("accuracy_drop", 0))
                    fr_vals.append(met.get("flip_rate", 0))
                    ss_vals.append(met.get("sensitivity_score", 0))
                    dsem_vals.append(met.get("semantic_distance", 0))
            avg_dacc = sum(dacc_vals)/len(dacc_vals) if dacc_vals else float("nan")
            avg_fr = sum(fr_vals)/len(fr_vals) if fr_vals else float("nan")
            avg_ss = sum(ss_vals)/len(ss_vals) if ss_vals else float("nan")
            avg_dsem = sum(dsem_vals)/len(dsem_vals) if dsem_vals else float("nan")
            line += f" {avg_dacc:>+10.2f} {avg_fr:>8.3f} {avg_ss:>8.2f} {avg_dsem:>8.4f}"
        print(line)

print("\nSummary complete.")



Task: MNLI — Instruct models
  Perturbation           Cat           LLAMA3.1 acc LLAMA3.1 dacc LLAMA3.1 FR LLAMA3.1 SS LLAMA3.1 d_sem  GEMMA2 acc GEMMA2 dacc GEMMA2 FR GEMMA2 SS GEMMA2 d_sem  QWEN2.5 acc QWEN2.5 dacc QWEN2.5 FR QWEN2.5 SS QWEN2.5 d_sem
--------------------------------------------------------------------------------------------------------------
  ctx_irrelevant               44.3     +0.0  0.080     0.00  0.0611      59.7     -1.3  0.053     0.07  0.0611      80.3     +0.0  0.033     0.00  0.0611  [contextual]
  ctx_role                     45.3     -1.0  0.290     0.29  0.0361      52.3     +6.0  0.147     0.88  0.0361      82.0     -1.7  0.053     0.09  0.0361  [contextual]
  fmt_caps                     39.3     +5.0  0.100     0.50  0.0000 !      55.0     +3.3  0.130     0.43  0.0000      83.3     -3.0  0.087     0.26  0.0000  [formatting]
  fmt_separators               52.0     -7.7  0.100     0.77  0.0000      57.7     +0.7  0.047     0.03  0.0000      84.0     

## Cell 19a - Export Flat Files for Plotting

[v6] CSV exports now use sensitivity_score instead of FI.
Added direction column. Added perturbation audit CSV.


In [30]:
rows_all = []
rows_cat = defaultdict(lambda: defaultdict(list))

for model_key in INSTRUCT_MODELS:
    s = load_summary(model_key)
    for task in ["mnli", "bbh_date", "arc"]:
        for pert, met in s.get(task, {}).items():
            if pert == "baseline":
                continue
            ss = met.get("sensitivity_score", 0)
            cat = met.get("category", "")
            rows_all.append({
                "model":            model_key,
                "task":             task,
                "perturbation":     pert,
                "category":         cat,
                "base_acc":         met.get("base_accuracy", 0),
                "pert_acc":         met.get("pert_accuracy", 0),
                "acc_drop":         met.get("accuracy_drop", 0),
                "flip_rate":        met.get("flip_rate", 0),
                "d_sem":            met.get("semantic_distance", 0),
                "sensitivity_score": ss,
                "direction":        met.get("direction", ""),
                "collapsed":        met.get("collapsed", False),
            })
            rows_cat[(model_key, task, cat)]["acc_drop"].append(met.get("accuracy_drop", 0))
            rows_cat[(model_key, task, cat)]["abs_acc_drop"].append(abs(met.get("accuracy_drop", 0)))
            rows_cat[(model_key, task, cat)]["flip_rate"].append(met.get("flip_rate", 0))
            rows_cat[(model_key, task, cat)]["SS"].append(ss)
            rows_cat[(model_key, task, cat)]["d_sem"].append(met.get("semantic_distance", 0))

df_all = pd.DataFrame(rows_all)
df_all.to_csv("results/all_metrics.csv", index=False)
print(f"Saved results/all_metrics.csv ({len(df_all)} rows)")

cat_rows = []
for (model_key, task, cat), vals in rows_cat.items():
    cat_rows.append({
        "model":              model_key,
        "task":               task,
        "category":           cat,
        "avg_signed_acc_drop": round(sum(vals["acc_drop"])/len(vals["acc_drop"]), 3),
        "avg_abs_acc_drop":    round(sum(vals["abs_acc_drop"])/len(vals["abs_acc_drop"]), 3),
        "avg_flip_rate":       round(sum(vals["flip_rate"])/len(vals["flip_rate"]), 4),
        "avg_sensitivity_score": round(sum(vals["SS"])/len(vals["SS"]), 3),
        "avg_semantic_distance": round(sum(vals["d_sem"])/len(vals["d_sem"]), 6),
    })
df_cat = pd.DataFrame(cat_rows)
df_cat.to_csv("results/category_summary.csv", index=False)
print(f"Saved results/category_summary.csv ({len(df_cat)} rows)")

# [v6] Perturbation audit CSV
audit_rows = []
for task, cache in TASK_CACHE.items():
    audit_rows.extend(export_perturbation_audit(cache, task))
df_audit = pd.DataFrame(audit_rows)
df_audit.to_csv("results/prompt_templates.csv", index=False)
print(f"Saved results/prompt_templates.csv ({len(df_audit)} rows)")

print("\nPreview of all_metrics.csv:")
print(df_all.head(10).to_string(index=False))


Saved results/all_metrics.csv (81 rows)
Saved results/category_summary.csv (45 rows)
Saved results/prompt_templates.csv (27 rows)

Preview of all_metrics.csv:
               model     task        perturbation   category  base_acc  pert_acc  acc_drop  flip_rate    d_sem  sensitivity_score   direction  collapsed
llama3.1-8b-instruct     mnli         lex_rewrite    lexical     44.33     63.33    -19.00     0.4033 0.111398             7.6633 improvement      False
llama3.1-8b-instruct     mnli            lex_typo    lexical     44.33     41.33      3.00     0.3200 0.503901             0.9600        drop      False
llama3.1-8b-instruct     mnli       lex_lowercase    lexical     44.33     50.00     -5.67     0.0700 0.000000             0.3967 improvement      False
llama3.1-8b-instruct     mnli   struct_inst_first structural     44.33     57.67    -13.33     0.4000 0.000000             5.3333 improvement      False
llama3.1-8b-instruct     mnli            ctx_role contextual     44.33     4

## Cell 20 - Instruct vs Base Comparison

[v6] Uses sensitivity_score instead of FI.
Base models flagged with competence warnings where baseline accuracy ≈ chance level.


In [31]:
PAIRS = [
    ("llama3.1-8b-instruct", "llama3.1-8b-base", "Llama-3.1-8B"),
    ("gemma2-9b-it",         "gemma2-9b-base",   "Gemma-2-9B"),
    ("qwen2.5-7b-instruct",  "qwen2.5-7b-base",  "Qwen2.5-7B"),
]

# Chance levels per task for competence gating
# Use empirical chance on the exact sampled subsets for tasks with variable
# numbers of answer choices.
_bbh_chance = sum(1.0 / len(ex["candidate_labels"]) for ex in bbh_data) / len(bbh_data) * 100
_arc_chance = sum(1.0 / len(ex["candidate_labels"]) for ex in arc_data) / len(arc_data) * 100

CHANCE_LEVEL = {
    "mnli": 100.0 / 3.0,
    "bbh_date": _bbh_chance,
    "arc": _arc_chance,
}

print(f"Empirical BBH chance level: {_bbh_chance:.1f}% (from {len(bbh_data)} examples)")
print(f"Empirical ARC chance level: {_arc_chance:.1f}% (from {len(arc_data)} examples)")

for task in ["mnli", "bbh_date", "arc"]:
    print(f"\n{'='*80}")
    print(f"Task: {task.upper()} — Instruct vs Base")
    print(f"{'Family':<14} {'Type':<10} {'Base acc':>9} {'Avg SS':>9} {'Avg dacc':>10} {'Status':>10}")
    print("-"*65)

    for instruct_key, base_key, label in PAIRS:
        for model_key, model_type in [(instruct_key, "instruct"), (base_key, "base")]:
            s = load_summary(model_key)
            task_data = s.get(task, {})
            if not task_data:
                print(f"  {label:<14} {model_type:<10} no data")
                continue

            baseline_acc = task_data.get("baseline", {}).get("base_accuracy", float("nan"))
            ss_vals, dacc_vals = [], []

            for pert, met in task_data.items():
                if pert == "baseline":
                    continue
                ss_vals.append(met.get("sensitivity_score", 0))
                dacc_vals.append(met.get("accuracy_drop", 0))

            avg_ss = sum(ss_vals) / len(ss_vals) if ss_vals else float("nan")
            avg_dacc = sum(dacc_vals) / len(dacc_vals) if dacc_vals else float("nan")

            chance = CHANCE_LEVEL.get(task, 25.0)
            status = "⚠ CHANCE" if baseline_acc <= chance + 5 else "✓"

            print(
                f"  {label:<14} {model_type:<10} {baseline_acc:>9.1f} "
                f"{avg_ss:>9.2f} {avg_dacc:>+10.2f} {status:>10}"
            )
    print()

Empirical BBH chance level: 17.2% (from 250 examples)
Empirical ARC chance level: 25.0% (from 300 examples)

Task: MNLI — Instruct vs Base
Family         Type        Base acc    Avg SS   Avg dacc     Status
-----------------------------------------------------------------
  Llama-3.1-8B   instruct        44.3      2.47      -6.11          ✓
  Llama-3.1-8B   base            33.3      0.02      +0.33   ⚠ CHANCE
  Gemma-2-9B     instruct        58.3      0.80      -1.00          ✓
  Gemma-2-9B     base            33.3      0.03      -0.22   ⚠ CHANCE
  Qwen2.5-7B     instruct        80.3      0.28      -2.67          ✓
  Qwen2.5-7B     base            40.7      3.37      -9.22          ✓


Task: BBH_DATE — Instruct vs Base
Family         Type        Base acc    Avg SS   Avg dacc     Status
-----------------------------------------------------------------
  Llama-3.1-8B   instruct        46.0      0.45      +1.96          ✓
  Llama-3.1-8B   base            18.0      0.05      -0.84   ⚠ CHAN

## Cell 21 - Cross-Model Agreement

[REVIEW Fix 7] Spearman added as primary statistic alongside Pearson.
[v6] Updated to work with new perturbation names (9 perturbations after removing struct_inst_last).


In [32]:
from scipy.stats import pearsonr, spearmanr
import itertools

print("Cross-model agreement (instruct models)\n")

for task in ["mnli", "bbh_date", "arc"]:
    print(f"  {task.upper()}")
    summaries = {m: load_summary(m) for m in INSTRUCT_MODELS}
    model_drops = {}
    for m in INSTRUCT_MODELS:
        task_data = summaries.get(m, {}).get(task, {})
        pert_names = sorted([k for k in task_data if k != "baseline"])
        model_drops[m] = [task_data[pn].get("accuracy_drop", 0) for pn in pert_names]

    for m1, m2 in itertools.combinations(INSTRUCT_MODELS, 2):
        d1, d2 = model_drops[m1], model_drops[m2]
        if len(d1) > 1 and len(d1) == len(d2):
            r_s, p_s = spearmanr(d1, d2)
            r_p, p_p = pearsonr(d1, d2)
            n1 = m1.split("-")[0]
            n2 = m2.split("-")[0]
            print(f"    {n1} vs {n2}: Spearman rho={r_s:.3f} (p={p_s:.3f}) | Pearson r={r_p:.3f} (p={p_p:.3f})")
        else:
            print(f"    Insufficient data for {m1} vs {m2}")
    print()

Cross-model agreement (instruct models)

  MNLI
    llama3.1 vs gemma2: Spearman rho=0.733 (p=0.025) | Pearson r=0.606 (p=0.084)
    llama3.1 vs qwen2.5: Spearman rho=0.544 (p=0.130) | Pearson r=0.617 (p=0.077)
    gemma2 vs qwen2.5: Spearman rho=0.268 (p=0.486) | Pearson r=0.119 (p=0.760)

  BBH_DATE
    llama3.1 vs gemma2: Spearman rho=0.124 (p=0.751) | Pearson r=0.237 (p=0.538)
    llama3.1 vs qwen2.5: Spearman rho=0.008 (p=0.983) | Pearson r=0.101 (p=0.796)
    gemma2 vs qwen2.5: Spearman rho=-0.381 (p=0.311) | Pearson r=-0.360 (p=0.341)

  ARC
    llama3.1 vs gemma2: Spearman rho=-0.042 (p=0.914) | Pearson r=0.068 (p=0.861)
    llama3.1 vs qwen2.5: Spearman rho=0.286 (p=0.456) | Pearson r=0.386 (p=0.304)
    gemma2 vs qwen2.5: Spearman rho=0.435 (p=0.242) | Pearson r=0.441 (p=0.234)



## Cell 22 - Bootstrap CIs and McNemar Test

[REVIEW Fix 6] Statistical significance for accuracy changes.
[v6] Enhanced output: n_discordant, direction, underpowered flag (n_discordant < 20).
Updated to use new perturbation names.


In [33]:
from scipy.stats import chi2

_boot_rng = np.random.default_rng(SEED)


def bootstrap_ci(binary_array: list, n_boot: int = 2000) -> Tuple[float, float]:
    arr = np.array(binary_array, dtype=float)
    n = len(arr)
    boot_means = [
        _boot_rng.choice(arr, size=n, replace=True).mean()
        for _ in range(n_boot)
    ]
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return float(lo), float(hi)


def paired_bootstrap_ci(base_correct: list, pert_correct: list,
                         n_boot: int = 2000) -> Tuple[float, float]:
    base = np.array(base_correct, dtype=float)
    pert = np.array(pert_correct, dtype=float)
    n = len(base)
    boot_drops = []
    for _ in range(n_boot):
        idx = _boot_rng.integers(0, n, size=n)
        drop = (base[idx].mean() - pert[idx].mean()) * 100
        boot_drops.append(drop)
    lo, hi = np.percentile(boot_drops, [2.5, 97.5])
    return float(lo), float(hi)


def mcnemar_test(base_correct: list, pert_correct: list) -> dict:
    """
    [v6] Returns dict with p-value, b, c, n_discordant for richer reporting.
    """
    b = sum(1 for bc, pc in zip(base_correct, pert_correct) if bc and not pc)
    c = sum(1 for bc, pc in zip(base_correct, pert_correct) if not bc and pc)
    n_disc = b + c
    if n_disc == 0:
        p = 1.0
    else:
        # [v7.1 Fix S1] max(0, ...) prevents squaring a negative Yates remainder
        stat = (max(0, abs(b - c) - 1)) ** 2 / (b + c)
        p = float(1 - chi2.cdf(stat, df=1))
    return {"p": p, "b": b, "c": c, "n_discordant": n_disc}


def holm_bonferroni(p_values: list) -> list:
    n = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [None] * n
    cummax = 0.0
    for rank, (orig_idx, p) in enumerate(indexed):
        adj = min(p * (n - rank), 1.0)
        cummax = max(cummax, adj)
        adjusted[orig_idx] = cummax
    return adjusted


stat_rows = []
all_p_values = []

for model_key in INSTRUCT_MODELS:
    for task in ["mnli", "bbh_date", "arc"]:
        base_path = f"results/{model_key}/{task}_baseline.json"
        if not os.path.exists(base_path):
            print(f"Missing baseline: {base_path}")
            continue
        with open(base_path) as f:
            baseline_results = json.load(f)
        base_correct = [r["correct"] for r in baseline_results]

        base_lo, base_hi = bootstrap_ci(base_correct)
        base_acc = sum(base_correct) / len(base_correct) * 100
        print(f"\n{model_key} | {task.upper()}")
        print(f"  Baseline acc: {base_acc:.1f}%  95% CI: [{base_lo*100:.1f}, {base_hi*100:.1f}]")

        for pert_name in ALL_PERT_NAMES:
            if pert_name == "baseline":
                continue
            pert_path = f"results/{model_key}/{task}_{pert_name}.json"
            if not os.path.exists(pert_path):
                continue
            with open(pert_path) as f:
                pert_data = json.load(f)
            pert_results = pert_data["pert_results"]
            pert_correct = [r["correct"] for r in pert_results]

            pert_acc = sum(pert_correct) / len(pert_correct) * 100
            drop = base_acc - pert_acc
            drop_lo, drop_hi = paired_bootstrap_ci(base_correct, pert_correct)
            mc = mcnemar_test(base_correct, pert_correct)
            all_p_values.append(mc["p"])

            # Direction
            if drop > 0:
                direction = "drop"
            elif drop < 0:
                direction = "improvement"
            else:
                direction = "no_change"

            underpowered = mc["n_discordant"] < 20

            print(f"  {pert_name:<22} dacc={drop:>+6.1f}pp  "
                  f"CI:[{drop_lo:>+5.1f},{drop_hi:>+5.1f}]  "
                  f"McNemar p={mc['p']:.4f}  "
                  f"b={mc['b']} c={mc['c']}"
                  f"{'  UNDERPOWERED' if underpowered else ''}")

            stat_rows.append({
                "model":        model_key,
                "task":         task,
                "perturbation": pert_name,
                "base_acc":     round(base_acc, 2),
                "base_ci_lo":   round(base_lo * 100, 2),
                "base_ci_hi":   round(base_hi * 100, 2),
                "pert_acc":     round(pert_acc, 2),
                "acc_drop":     round(drop, 2),
                "drop_ci_lo":   round(drop_lo, 2),
                "drop_ci_hi":   round(drop_hi, 2),
                "mcnemar_p":    round(mc["p"], 6),
                "b":            mc["b"],
                "c":            mc["c"],
                "n_discordant": mc["n_discordant"],
                "direction":    direction,
                "underpowered": underpowered,
            })

if stat_rows:
    df_stats = pd.DataFrame(stat_rows)

    adjusted = holm_bonferroni(all_p_values)
    df_stats["mcnemar_p_adj"] = [round(a, 6) for a in adjusted]
    df_stats["significant_005"] = df_stats["mcnemar_p_adj"] < 0.05

    df_stats.to_csv("results/stats.csv", index=False)
    n_sig = df_stats["significant_005"].sum()
    n_under = df_stats["underpowered"].sum()
    print(f"\nSaved results/stats.csv ({len(df_stats)} rows)")
    print(f"Holm-Bonferroni correction applied across {len(all_p_values)} tests.")
    print(f"  Significant at alpha=0.05 after correction: {n_sig}/{len(all_p_values)}")
    print(f"  Underpowered tests (n_discordant < 20): {n_under}/{len(all_p_values)}")

    # Summary by direction
    sig_drops = df_stats[(df_stats["significant_005"]) & (df_stats["direction"] == "drop")]
    sig_improv = df_stats[(df_stats["significant_005"]) & (df_stats["direction"] == "improvement")]
    print(f"\n  Significant HARMFUL perturbations: {len(sig_drops)}")
    for _, r in sig_drops.iterrows():
        print(f"    {r['model'][:20]} | {r['task']:8} | {r['perturbation']:22} | dacc={r['acc_drop']:+.1f}")
    print(f"  Significant BENEFICIAL perturbations: {len(sig_improv)}")
    for _, r in sig_improv.iterrows():
        print(f"    {r['model'][:20]} | {r['task']:8} | {r['perturbation']:22} | dacc={r['acc_drop']:+.1f}")



llama3.1-8b-instruct | MNLI
  Baseline acc: 44.3%  95% CI: [39.0, 49.3]
  lex_rewrite            dacc= -19.0pp  CI:[-24.7,-13.0]  McNemar p=0.0000  b=18 c=75
  lex_typo               dacc=  +3.0pp  CI:[ -2.0, +8.0]  McNemar p=0.3135  b=36 c=27
  lex_lowercase          dacc=  -5.7pp  CI:[ -8.7, -3.0]  McNemar p=0.0002  b=1 c=18  UNDERPOWERED
  struct_inst_first      dacc= -13.3pp  CI:[-19.3, -7.3]  McNemar p=0.0000  b=21 c=61
  ctx_role               dacc=  -1.0pp  CI:[ -5.7, +4.0]  McNemar p=0.7911  b=27 c=30
  ctx_irrelevant         dacc=  +0.0pp  CI:[ -3.0, +3.0]  McNemar p=1.0000  b=10 c=10
  fmt_caps               dacc=  +5.0pp  CI:[ +1.7, +8.3]  McNemar p=0.0071  b=21 c=6
  fmt_separators         dacc=  -7.7pp  CI:[-11.0, -4.7]  McNemar p=0.0000  b=2 c=25
  instruction_rewrite    dacc= -16.3pp  CI:[-22.0,-10.7]  McNemar p=0.0000  b=18 c=67

llama3.1-8b-instruct | BBH_DATE
  Baseline acc: 46.0%  95% CI: [40.0, 52.0]
  lex_rewrite            dacc=  +1.2pp  CI:[ -0.4, +3.2]  McNemar

## Cell 23 - Qualitative Error Analysis

[REVIEW Fix 9] Two types of failures:
1. Flip failures: baseline correct, perturbation wrong
2. Confidence drops: both correct, but margin shrank by > 0.5 nats

Margin = top logprob - second logprob. A drop in margin even without an
accuracy change shows the model became less certain, which is a form of
fragility not captured by accuracy alone.

In [34]:
def show_error_analysis(model_key: str, task: str, pert_name: str, n_show: int = 3):
    """
    Load paired results from disk and print:
    1. Flip failures (baseline correct, pert wrong)
    2. Confidence drops (both correct but margin dropped > 0.5 nats)
    """
    base_path = f"results/{model_key}/{task}_baseline.json"
    pert_path = f"results/{model_key}/{task}_{pert_name}.json"
    if not os.path.exists(base_path) or not os.path.exists(pert_path):
        print(f"Results not found for {model_key}/{task}/{pert_name}")
        return

    with open(base_path) as f:
        base_results = json.load(f)
    with open(pert_path) as f:
        pert_data = json.load(f)
    pert_results = pert_data["pert_results"]

    print(f"\n{model_key} | {task} | {pert_name}")
    print(f"{'='*60}")

    # Flip failures
    flips = [
        (b, p) for b, p in zip(base_results, pert_results)
        if b["correct"] and not p["correct"]
    ]
    print(f"\nFlip failures (baseline correct, pert wrong): {len(flips)} total")
    for b, p in flips[:n_show]:
        print(f"  idx={b['idx']} gold={b['gold_label']} "
              f"base={b['predicted_label']}(margin={b.get('margin',0):.2f}) "
              f"pert={p['predicted_label']}(margin={p.get('margin',0):.2f})")

    # Confidence drops (both correct, margin shrank)
    conf_drops = [
        (b, p) for b, p in zip(base_results, pert_results)
        if b["correct"] and p["correct"]
        and (b.get("margin", 0) - p.get("margin", 0)) > 0.5
    ]
    print(f"\nConfidence drops > 0.5 nats (both correct): {len(conf_drops)} total")
    for b, p in conf_drops[:n_show]:
        drop = b.get("margin", 0) - p.get("margin", 0)
        print(f"  idx={b['idx']} gold={b['gold_label']} "
              f"base_margin={b.get('margin',0):.2f} "
              f"pert_margin={p.get('margin',0):.2f} "
              f"drop={drop:.2f}")


# [v6] Updated perturbation names
for model_key in ["llama3.1-8b-instruct", "gemma2-9b-it"]:
    for task in ["mnli", "arc"]:
        show_error_analysis(model_key, task, "struct_inst_first")
        show_error_analysis(model_key, task, "lex_rewrite")



llama3.1-8b-instruct | mnli | struct_inst_first

Flip failures (baseline correct, pert wrong): 21 total
  idx=6 gold=A base=A(margin=0.34) pert=B(margin=0.28)
  idx=43 gold=A base=A(margin=2.13) pert=B(margin=0.11)
  idx=49 gold=C base=C(margin=0.59) pert=B(margin=0.09)

Confidence drops > 0.5 nats (both correct): 33 total
  idx=1 gold=A base_margin=3.73 pert_margin=2.62 drop=1.11
  idx=36 gold=A base_margin=4.39 pert_margin=1.83 drop=2.56
  idx=40 gold=A base_margin=4.09 pert_margin=3.58 drop=0.51

llama3.1-8b-instruct | mnli | lex_rewrite

Flip failures (baseline correct, pert wrong): 18 total
  idx=6 gold=A base=A(margin=0.34) pert=B(margin=1.16)
  idx=53 gold=A base=A(margin=1.73) pert=B(margin=0.20)
  idx=62 gold=A base=A(margin=0.05) pert=B(margin=0.97)

Confidence drops > 0.5 nats (both correct): 81 total
  idx=0 gold=A base_margin=2.14 pert_margin=0.36 drop=1.78
  idx=1 gold=A base_margin=3.73 pert_margin=1.30 drop=2.44
  idx=18 gold=A base_margin=2.97 pert_margin=0.56 drop=2.

## Cell 24 - Chat Template Sanity Check

[v6] Now that format_prompt uses tokenizer.apply_chat_template natively,
this cell should show 0 disagreements for all models. If it doesn't,
there's a tokenizer configuration issue.


In [35]:
# [v6.1] jinja2 install moved to Cell 1 (main dependencies).
# No separate install or kernel restart needed here.
print("jinja2 already installed with main dependencies.")

jinja2 already installed with main dependencies.


In [36]:
N_CHAT_CHECK = 50
N_GREEDY_CHECK = 20

TASKS_FOR_CHECK = {
    "mnli":     (mnli_data,  ["A", "B", "C"],  make_mnli_prompt),
    "bbh_date": (bbh_data,   None,             make_bbh_prompt),
    "arc":      (arc_data,   None,             make_arc_prompt),
}

print(f"Chat template sanity check: {N_CHAT_CHECK} examples x 3 tasks x 3 models")
print("format_and_tokenize uses tokenize=True; this checks it matches")
print("a standalone apply_chat_template(tokenize=True) call.")
print(f"A separate 1-step greedy smoke test runs on {N_GREEDY_CHECK} examples per task/model.\n")

for model_key in INSTRUCT_MODELS:
    model, tokenizer = load_model(model_key)

    for task, (data, fixed_labels, make_prompt) in TASKS_FOR_CHECK.items():
        pipeline_correct = 0
        standalone_correct = 0
        disagree = 0
        check_data = data[:N_CHAT_CHECK]

        for ex in check_data:
            labels = fixed_labels if fixed_labels else ex["candidate_labels"]
            gold = ex["gold_label"]
            user_message = make_prompt(ex)

            # Pipeline path
            result_pipeline = get_label_logprobs(
                model, tokenizer, model_key,
                user_message, labels, gold,
            )

            # Standalone chat-template path
            messages = [{"role": "user", "content": user_message}]
            templated = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            input_ids = _extract_input_ids_tensor(templated).to(model.device)

            with torch.no_grad():
                outputs = model(input_ids=input_ids, use_cache=False)
                log_probs = torch.log_softmax(outputs.logits[0, -1, :], dim=-1)

            standalone_prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            standalone_prompt_ids = input_ids[0].detach().cpu().tolist()

            lp = {}
            for label in labels:
                cont_ids = continuation_token_ids(
                    tokenizer,
                    label,
                    prompt_ids=standalone_prompt_ids,
                    prompt_str=standalone_prompt,
                )
                lp[label] = log_probs[cont_ids[0]].item()

            pred_standalone = max(lp, key=lp.get)

            if result_pipeline.correct:
                pipeline_correct += 1
            if pred_standalone == gold:
                standalone_correct += 1
            if result_pipeline.predicted_label != pred_standalone:
                disagree += 1

        n = len(check_data)
        status = "✓" if disagree == 0 else f"⚠ {disagree} DISAGREEMENTS"
        print(
            f"{model_key} | {task}: pipeline={pipeline_correct}/{n}  "
            f"standalone={standalone_correct}/{n}  disagree={disagree}/{n}  {status}"
        )

        # Greedy smoke test:
        # decode the top next token and compare it to the logprob-scored label.
        # This is a smoke test, not a proof.
        greedy_checked = 0
        greedy_skipped = 0
        greedy_mismatches = []

        for ex in data[:N_GREEDY_CHECK]:
            labels = fixed_labels if fixed_labels else ex["candidate_labels"]
            gold = ex["gold_label"]
            user_message = make_prompt(ex)

            scored = get_label_logprobs(
                model, tokenizer, model_key,
                user_message, labels, gold,
            )

            prompt_ids, prompt_str = format_and_tokenize(
                model_key, user_message, tokenizer=tokenizer
            )
            prompt_ids = _extract_input_ids_tensor(prompt_ids).to(model.device)

            with torch.no_grad():
                outputs = model(input_ids=prompt_ids, use_cache=False)
                greedy_id = int(torch.argmax(outputs.logits[0, -1, :]).item())

            greedy_piece = tokenizer.decode([greedy_id], skip_special_tokens=True)
            greedy_norm = greedy_piece.strip()

            if greedy_norm in labels:
                greedy_checked += 1
                if greedy_norm != scored.predicted_label:
                    greedy_mismatches.append({
                        "gold": gold,
                        "greedy_raw": greedy_piece,
                        "greedy_norm": greedy_norm,
                        "scored": scored.predicted_label,
                        "prompt_preview": prompt_str[:120].replace("\n", "\\n"),
                    })
            else:
                greedy_skipped += 1

        if greedy_mismatches:
            print(
                f"  greedy smoke test: ⚠ {len(greedy_mismatches)}/{greedy_checked} mismatches "
                f"(skipped {greedy_skipped} non-label greedy tokens)"
            )
            for mm in greedy_mismatches[:3]:
                print(
                    f"    gold={mm['gold']} greedy_raw={mm['greedy_raw']!r} "
                    f"greedy_norm={mm['greedy_norm']!r} scored={mm['scored']} "
                    f"prompt={mm['prompt_preview']}..."
                )
        else:
            print(
                f"  greedy smoke test: ✓ 0/{greedy_checked} mismatches "
                f"(skipped {greedy_skipped} non-label greedy tokens)"
            )

    clear_model()

Chat template sanity check: 50 examples x 3 tasks x 3 models
format_and_tokenize uses tokenize=True; this checks it matches
a standalone apply_chat_template(tokenize=True) call.
A separate 1-step greedy smoke test runs on 20 examples per task/model.

No model loaded.
Loading llama3.1-8b-instruct from meta-llama/Meta-Llama-3.1-8B-Instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded in 80s, VRAM used: 4.6 GB
llama3.1-8b-instruct | mnli: pipeline=16/50  standalone=16/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
llama3.1-8b-instruct | bbh_date: pipeline=20/50  standalone=20/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/6 mismatches (skipped 14 non-label greedy tokens)
llama3.1-8b-instruct | arc: pipeline=45/50  standalone=45/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/10 mismatches (skipped 10 non-label greedy tokens)
Clearing llama3.1-8b-instruct from memory.
GPU freed. Available: 81.6 GB
No model loaded.
Loading gemma2-9b-it from google/gemma-2-9b-it


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Loaded in 138s, VRAM used: 5.4 GB
gemma2-9b-it | mnli: pipeline=32/50  standalone=32/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
gemma2-9b-it | bbh_date: pipeline=33/50  standalone=33/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/19 mismatches (skipped 1 non-label greedy tokens)
gemma2-9b-it | arc: pipeline=46/50  standalone=46/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
Clearing gemma2-9b-it from memory.
GPU freed. Available: 79.8 GB
No model loaded.
Loading qwen2.5-7b-instruct from Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded in 63s, VRAM used: 6.5 GB
qwen2.5-7b-instruct | mnli: pipeline=41/50  standalone=41/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
qwen2.5-7b-instruct | bbh_date: pipeline=35/50  standalone=35/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
qwen2.5-7b-instruct | arc: pipeline=46/50  standalone=46/50  disagree=0/50  ✓
  greedy smoke test: ✓ 0/20 mismatches (skipped 0 non-label greedy tokens)
Clearing qwen2.5-7b-instruct from memory.
GPU freed. Available: 80.6 GB


In [37]:
import os
import zipfile

def create_export_zip(zip_filename="hcnlp_experiment_data.zip", dirs_to_zip=["results", "perturbation_cache"]):
    """Zips the specified directories into a single downloadable archive."""
    print(f"Creating {zip_filename}...")
    
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for dir_path in dirs_to_zip:
            if not os.path.exists(dir_path):
                print(f"  ⚠ Warning: Directory '{dir_path}' not found. Skipping.")
                continue
                
            # Walk through the directory and add all files
            file_count = 0
            for root, dirs, files in os.walk(dir_path):
                for file in files:
                    file_path = os.path.join(root, file)
                    # Keep the directory structure clean inside the zip (e.g., results/...)
                    arcname = os.path.relpath(file_path, start=os.path.dirname(os.path.abspath(dir_path)))
                    zipf.write(file_path, arcname)
                    file_count += 1
            print(f"  ✓ Added {file_count} files from '{dir_path}'")
            
    print(f"\nDone! Your data is ready to download: {zip_filename}")
    
    # Print file size for a quick sanity check
    size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
    print(f"Archive size: {size_mb:.2f} MB")

# Run the export
create_export_zip()

Creating hcnlp_experiment_data.zip...
  ✓ Added 352 files from 'results'
  ✓ Added 3 files from 'perturbation_cache'

Done! Your data is ready to download: hcnlp_experiment_data.zip
Archive size: 6.24 MB


In [ ]:
Analysis only (below)

In [40]:
import pandas as pd

df = pd.read_csv("results/all_metrics.csv")
print(df.head())
print(df.shape)

                  model  task       perturbation    category  base_acc  \
0  llama3.1-8b-instruct  mnli        lex_rewrite     lexical     44.33   
1  llama3.1-8b-instruct  mnli           lex_typo     lexical     44.33   
2  llama3.1-8b-instruct  mnli      lex_lowercase     lexical     44.33   
3  llama3.1-8b-instruct  mnli  struct_inst_first  structural     44.33   
4  llama3.1-8b-instruct  mnli           ctx_role  contextual     44.33   

   pert_acc  acc_drop  flip_rate     d_sem  sensitivity_score    direction  \
0     63.33    -19.00     0.4033  0.111398             7.6633  improvement   
1     41.33      3.00     0.3200  0.503901             0.9600         drop   
2     50.00     -5.67     0.0700  0.000000             0.3967  improvement   
3     57.67    -13.33     0.4000  0.000000             5.3333  improvement   
4     45.33     -1.00     0.2900  0.036142             0.2900  improvement   

   collapsed  
0      False  
1      False  
2      False  
3      False  
4      Fals

In [41]:
m = df[(df["model"] == "llama3.1-8b-instruct") & (df["task"] == "mnli")]
print(m[["perturbation","base_acc","pert_acc","acc_drop","flip_rate","d_sem","sensitivity_score","direction","collapsed"]]
      .sort_values("perturbation")
      .to_string(index=False))

       perturbation  base_acc  pert_acc  acc_drop  flip_rate    d_sem  sensitivity_score   direction  collapsed
     ctx_irrelevant     44.33     44.33      0.00     0.0800 0.061145             0.0000   no_change      False
           ctx_role     44.33     45.33     -1.00     0.2900 0.036142             0.2900 improvement      False
           fmt_caps     44.33     39.33      5.00     0.1000 0.000000             0.5000        drop       True
     fmt_separators     44.33     52.00     -7.67     0.1000 0.000000             0.7667 improvement      False
instruction_rewrite     44.33     60.67    -16.33     0.3867 0.148178             6.3156 improvement      False
      lex_lowercase     44.33     50.00     -5.67     0.0700 0.000000             0.3967 improvement      False
        lex_rewrite     44.33     63.33    -19.00     0.4033 0.111398             7.6633 improvement      False
           lex_typo     44.33     41.33      3.00     0.3200 0.503901             0.9600        drop    

In [42]:
pt = pd.read_csv("results/prompt_templates.csv")
mnli_pt = pt[pt["task"] == "mnli"]
print(mnli_pt[["perturbation","category","instruction_semantic_distance","closest_other_pert","min_pairwise_dsem"]]
      .sort_values("perturbation")
      .to_string(index=False))

       perturbation   category  instruction_semantic_distance  closest_other_pert  min_pairwise_dsem
     ctx_irrelevant contextual                       0.061145       lex_lowercase           0.061145
           ctx_role contextual                       0.036142       lex_lowercase           0.036142
           fmt_caps formatting                       0.000000       lex_lowercase           0.000000
     fmt_separators formatting                       0.000000       lex_lowercase           0.000000
instruction_rewrite    rewrite                       0.148178         lex_rewrite           0.055885
      lex_lowercase    lexical                       0.000000   struct_inst_first           0.000000
        lex_rewrite    lexical                       0.111398 instruction_rewrite           0.055885
           lex_typo    lexical                       0.347833            ctx_role           0.321455
  struct_inst_first structural                       0.000000       lex_lowercase          

In [43]:
for p in ["fmt_separators", "instruction_rewrite", "lex_rewrite", "lex_typo"]:
    row = mnli_pt[mnli_pt["perturbation"] == p].iloc[0]
    print("\nPERT:", p)
    print("d_sem:", row["instruction_semantic_distance"])
    print("base preview:", row["base_prompt_preview"][:180])
    print("pert preview:", row["perturbed_prompt_preview"][:180])


PERT: fmt_separators
d_sem: 0.0
base preview: Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated i
pert preview: Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion. | Hypothesis: Unfortunately his face had been mutilated

PERT: instruction_rewrite
d_sem: 0.148178
base preview: Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated i
pert preview: Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated i

PERT: lex_rewrite
d_sem: 0.111398
base preview: Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated i
pert preview: Premise: I touched my palm to his mutilated cheek,

In [44]:
import json

with open("results/llama3.1-8b-instruct/mnli_baseline.json") as f:
    base = json.load(f)

with open("results/llama3.1-8b-instruct/mnli_lex_rewrite.json") as f:
    lex = json.load(f)["pert_results"]

with open("results/llama3.1-8b-instruct/mnli_fmt_caps.json") as f:
    caps = json.load(f)["pert_results"]

with open("results/llama3.1-8b-instruct/mnli_ctx_irrelevant.json") as f:
    irr = json.load(f)["pert_results"]

with open("results/llama3.1-8b-instruct/mnli_instruction_rewrite.json") as f:
    rew = json.load(f)["pert_results"]

In [45]:
def show_example(i, pert_name, pert):
    print("\n" + "="*90)
    print("IDX:", i, "| PERT:", pert_name)
    print("GOLD:", base[i]["gold_label"])
    print("BASE PRED:", base[i]["predicted_label"], "| margin:", round(base[i]["margin"], 4))
    print("PERT PRED:", pert[i]["predicted_label"], "| margin:", round(pert[i]["margin"], 4))
    print("\nBASE PROMPT:\n", base[i]["prompt"][:800])
    print("\nPERT PROMPT:\n", pert[i]["prompt"][:800])

for i in [0, 1, 2]:
    show_example(i, "lex_rewrite", lex)


IDX: 0 | PERT: lex_rewrite
GOLD: A
BASE PRED: A | margin: 2.1412
PERT PRED: A | margin: 0.3589

BASE PROMPT:
 Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated in as least one way. 
Does the premise entail, contradict, or is it neutral to the hypothesis?
Answer with one letter only: A for entailment, B for neutral, C for contradiction.

PERT PROMPT:
 Premise: I touched my palm to his mutilated cheek, and tried to stem my instinctive revulsion.
Hypothesis: Unfortunately his face had been mutilated in as least one way. 
What is the relationship between the premise and the hypothesis?
Answer with one letter only: A for entailment, B for neutral, C for contradiction.

IDX: 1 | PERT: lex_rewrite
GOLD: A
BASE PRED: A | margin: 3.7342
PERT PRED: A | margin: 1.2971

BASE PROMPT:
 Premise: and the wind started blowing and it was one of my earlier trips to be really out in the middle of
Hypothesis

In [46]:
pair = pd.read_csv("results/paired/llama3.1-8b-instruct_mnli_lex_rewrite.csv")
print(pair.head())
print(pair.shape)

improved = pair[(pair["base_correct"] == 0) & (pair["pert_correct"] == 1)]
hurt = pair[(pair["base_correct"] == 1) & (pair["pert_correct"] == 0)]
same_flip = pair[(pair["flip"] == 1) & (pair["base_correct"] == pair["pert_correct"])]

print("improved:", len(improved))
print("hurt:", len(hurt))
print("flip but same correctness:", len(same_flip))

print("\nImprovement examples:")
print(improved.head(10).to_string(index=False))

   idx gold base_pred pert_pred  base_correct  pert_correct  flip  \
0    0    A         A         A             1             1     0   
1    1    A         A         A             1             1     0   
2    2    B         A         C             0             0     1   
3    3    C         A         C             0             1     1   
4    4    C         A         C             0             1     1   

   base_margin  pert_margin  
0       2.1412       0.3589  
1       3.7342       1.2971  
2       1.2815       0.2812  
3       0.0000       0.8591  
4       0.0625       1.7964  
(300, 9)
improved: 75
hurt: 18
flip but same correctness: 28

Improvement examples:
 idx gold base_pred pert_pred  base_correct  pert_correct  flip  base_margin  pert_margin
   3    C         A         C             0             1     1       0.0000       0.8591
   4    C         A         C             0             1     1       0.0625       1.7964
   5    C         A         C             0        

In [47]:
import json, pprint

for model in ["llama3.1-8b-base", "gemma2-9b-base", "qwen2.5-7b-base"]:
    with open(f"results/{model}/SUMMARY.json") as f:
        s = json.load(f)
    print("\nMODEL:", model)
    print("MNLI baseline:", s["mnli"]["baseline"]["base_accuracy"])
    print("BBH baseline:", s["bbh_date"]["baseline"]["base_accuracy"])
    print("ARC baseline:", s["arc"]["baseline"]["base_accuracy"])


MODEL: llama3.1-8b-base
MNLI baseline: 33.33
BBH baseline: 18.0
ARC baseline: 24.0

MODEL: gemma2-9b-base
MNLI baseline: 33.33
BBH baseline: 19.2
ARC baseline: 24.0

MODEL: qwen2.5-7b-base
MNLI baseline: 40.67
BBH baseline: 41.2
ARC baseline: 85.33


In [48]:
with open("results/qwen2.5-7b-base/SUMMARY.json") as f:
    qbase = json.load(f)

arc_rows = []
for pert, met in qbase["arc"].items():
    if pert == "baseline":
        continue
    arc_rows.append((pert, met["base_accuracy"], met["pert_accuracy"], met["accuracy_drop"], met["flip_rate"], met["semantic_distance"]))
print(sorted(arc_rows, key=lambda x: abs(x[3]), reverse=True)[:10])

[('ctx_role', 85.33, 24.67, 60.67, 0.7067, 0.399555), ('lex_typo', 85.33, 40.67, 44.67, 0.54, 0.584679), ('ctx_irrelevant', 85.33, 49.33, 36.0, 0.46, 0.384786), ('fmt_separators', 85.33, 59.67, 25.67, 0.33, 0.0), ('lex_lowercase', 85.33, 60.0, 25.33, 0.32, 0.0), ('fmt_caps', 85.33, 61.67, 23.67, 0.3567, 0.0), ('lex_rewrite', 85.33, 63.0, 22.33, 0.3167, 0.287199), ('struct_inst_first', 85.33, 67.0, 18.33, 0.31, 0.0), ('instruction_rewrite', 85.33, 71.67, 13.67, 0.2267, 0.338404)]


In [49]:
stats = pd.read_csv("results/stats.csv")
print(stats.head())
print(stats[["model","task","perturbation","acc_drop","drop_ci_lo","drop_ci_hi","mcnemar_p_adj","significant_005","underpowered"]]
      .sort_values(["significant_005","mcnemar_p_adj"], ascending=[False, True])
      .head(30)
      .to_string(index=False))

                  model  task       perturbation  base_acc  base_ci_lo  \
0  llama3.1-8b-instruct  mnli        lex_rewrite     44.33        39.0   
1  llama3.1-8b-instruct  mnli           lex_typo     44.33        39.0   
2  llama3.1-8b-instruct  mnli      lex_lowercase     44.33        39.0   
3  llama3.1-8b-instruct  mnli  struct_inst_first     44.33        39.0   
4  llama3.1-8b-instruct  mnli           ctx_role     44.33        39.0   

   base_ci_hi  pert_acc  acc_drop  drop_ci_lo  drop_ci_hi  mcnemar_p   b   c  \
0       49.33     63.33    -19.00      -24.67      -13.00   0.000000  18  75   
1       49.33     41.33      3.00       -2.00        8.00   0.313500  36  27   
2       49.33     50.00     -5.67       -8.67       -3.00   0.000242   1  18   
3       49.33     57.67    -13.33      -19.33       -7.33   0.000017  21  61   
4       49.33     45.33     -1.00       -5.67        4.00   0.791082  27  30   

   n_discordant    direction  underpowered  mcnemar_p_adj  significant_005

In [50]:
# 9 non-baseline perturbations per task per instruct model
print(df.groupby(["model","task"]).size())

# category summary should have 5 categories x 3 tasks x 3 instruct models = 45 rows
cat = pd.read_csv("results/category_summary.csv")
print(cat.shape)

# stats should have 9 perturbations x 3 tasks x 3 instruct models = 81 rows
print(stats.shape)

model                 task    
gemma2-9b-it          arc         9
                      bbh_date    9
                      mnli        9
llama3.1-8b-instruct  arc         9
                      bbh_date    9
                      mnli        9
qwen2.5-7b-instruct   arc         9
                      bbh_date    9
                      mnli        9
dtype: int64
(45, 8)
(81, 18)


In [51]:
import json, pandas as pd

with open("results/qwen2.5-7b-base/arc_baseline.json") as f:
    qb = json.load(f)

with open("results/qwen2.5-7b-base/arc_ctx_role.json") as f:
    qctx = json.load(f)["pert_results"]

with open("results/qwen2.5-7b-base/arc_lex_typo.json") as f:
    qtypo = json.load(f)["pert_results"]

pair_ctx = pd.read_csv("results/paired/qwen2.5-7b-base_arc_ctx_role.csv")
pair_typo = pd.read_csv("results/paired/qwen2.5-7b-base_arc_lex_typo.csv")

print(pair_ctx.head())
print(pair_typo.head())

   idx gold base_pred pert_pred  base_correct  pert_correct  flip  \
0    0    B         B         A             1             0     1   
1    1    C         C         A             1             0     1   
2    2    D         A         A             0             0     0   
3    3    B         D         A             0             0     1   
4    4    B         B         A             1             0     1   

   base_margin  pert_margin  
0       1.8125       3.7539  
1       1.7344       3.2422  
2       0.3594       2.7969  
3       0.8418       0.4453  
4       1.1270       3.1328  
   idx gold base_pred pert_pred  base_correct  pert_correct  flip  \
0    0    B         B         A             1             0     1   
1    1    C         C         C             1             1     0   
2    2    D         A         A             0             0     0   
3    3    B         D         A             0             0     1   
4    4    B         B         B             1             1 

In [52]:
def show_arc(i, name, pert):
    print("\n" + "="*90)
    print("IDX:", i, "| PERT:", name)
    print("GOLD:", qb[i]["gold_label"])
    print("BASE:", qb[i]["predicted_label"], qb[i]["correct"], "margin:", round(qb[i]["margin"], 4))
    print("PERT:", pert[i]["predicted_label"], pert[i]["correct"], "margin:", round(pert[i]["margin"], 4))
    print("\nBASE PROMPT:\n", qb[i]["prompt"][:900])
    print("\nPERT PROMPT:\n", pert[i]["prompt"][:900])

for i in pair_ctx[pair_ctx["flip"] == 1]["idx"].head(3):
    show_arc(int(i), "ctx_role", qctx)


IDX: 0 | PERT: ctx_role
GOLD: B
BASE: B True margin: 1.8125
PERT: A False margin: 3.7539

BASE PROMPT:
 Question: A group of engineers wanted to know how different building designs would respond during an earthquake. They made several models of buildings and tested each for its ability to withstand earthquake conditions. Which will most likely result from testing different building designs?
(A) buildings will be built faster
(B) buildings will be made safer
(C) building designs will look nicer
(D) building materials will be cheaper
Answer with only the letter of the correct choice.

PERT PROMPT:
 Question: A group of engineers wanted to know how different building designs would respond during an earthquake. They made several models of buildings and tested each for its ability to withstand earthquake conditions. Which will most likely result from testing different building designs?
(A) buildings will be built faster
(B) buildings will be made safer
(C) building designs will look nicer


In [53]:
import json, collections

def pred_dist(path):
    with open(path) as f:
        obj = json.load(f)
    if isinstance(obj, dict) and "pert_results" in obj:
        rows = obj["pert_results"]
    else:
        rows = obj
    c = collections.Counter(r["predicted_label"] for r in rows)
    n = len(rows)
    return {k: round(v / n, 4) for k, v in sorted(c.items())}

print("baseline:", pred_dist("results/qwen2.5-7b-base/arc_baseline.json"))
print("ctx_role:", pred_dist("results/qwen2.5-7b-base/arc_ctx_role.json"))
print("lex_typo:", pred_dist("results/qwen2.5-7b-base/arc_lex_typo.json"))

baseline: {'A': 0.2867, 'B': 0.2133, 'C': 0.2767, 'D': 0.2233}
ctx_role: {'A': 0.9933, 'D': 0.0067}
lex_typo: {'A': 0.8267, 'B': 0.0433, 'C': 0.0667, 'D': 0.0633}
